# Pupper zero-shot motion: keyframes vs. UniMate 🐾⏱️

**What this answers**, for three ways of turning a spoken request into a Pupper motion:

1. **Quality:** which one produces better motion?
2. **Latency:** how long from the request to a motion that is ready to play?

| | Option | How the motion is made |
|---|---|---|
| 1 | **Keyframes** | The LLM writes keyframes; they are interpolated (quintic, with "stop"/"flow" keyframes) and solved with the course IK. A **verifier** checks them (reach, speed, balance, sliding, physics) and sends the failures back to the LLM, up to 3 tries. |
| 2 | **Keyframes + UniMate** | The LLM writes a UniMate prompt **and decides how many keyframes it needs** (zero for a mood, several for "lift your left paw 5 cm for 1 s"). UniMate generates the motion and is forced through the keyframes (in-betweening). |
| 3 | **UniMate** | The LLM rewrites the request into UniMate prompts; UniMate generates; the best of N samples is cleaned up and fitted. |

**Fair by construction.** Everything after generation is the same code for all three: the same finishing
(blend in/out of a stand, slow down if a joint is too fast), the same **gate** (nothing in the floor, joints
under 15 rad/s, no slide over 3 cm, balanced, doesn't fall in a physics replay), and the same metrics. Each
option also gets its own best practice: a verifier loop for 1; best-of-N, ranking, foot locking, retries
with new seeds for 2 and 3; and section 6 tunes UniMate's settings before the comparison, so neither
option loses to a bad default.

**No tracker yet.** Motions are judged as references: kinematics, plain open-loop physics replay (as
`play_motion`), and your eyes. A tracking policy comes later and would make all three hold up better.

---

### How to run it

1. **Runtime → Change runtime type → G4** (RTX PRO 6000). UniMate needs it, and its latency only means something on the GPU you'd serve it from.
2. **Run sections 1-5 in order** (about 10 minutes, mostly installs). Every check in section 5 should say PASS; if one says FAIL, stop and look (or ask me) before trusting the numbers.
3. **Section 6 (recommended, ~10 min):** tune UniMate's settings, then put the best ones into `CFG` (section 4's Config).
4. **Section 7:** the benchmark. Pick an experiment name, the options, requests and repeats in the form, then run the next cell. All 12 requests × 3 options × 2 repeats ≈ 40-70 minutes. It saves after every run; rerunning continues where it stopped.
5. **Section 8:** pick which experiments to look at (merged; the newest wins for the same request, option and repeat), then the answers from the numbers: latency and objective quality.
6. **Section 9:** the viewer. Three Puppers side by side, **blind** (shuffled, all grey). Rate each, press Save. Ask a friend to rate too: blind ratings from someone who didn't build it are worth more.
7. **Section 10:** your ratings, summarized: the answer to "which looks better".

**LLM, at no cost:** by default the LLM is **Colab AI** (`google.colab.ai`), which is free with Colab and needs no API key;
Pro has higher limits. It serves Gemini models, so its latency is close to what the Gemini API would give.
To use your AI Studio credits instead, set `LLM_BACKEND = "aistudio"` and add `GEMINI_API_KEY` in Colab's **Secrets** (key icon).
The key is never printed or saved.

**What "latency" covers:** from the request text (what Gemini Live would hand a motion tool) to a 50 Hz motion that passed
the gate. Not included: the Live conversation turn before the tool call, network to the robot, and the robot's
controller start (~1.5 s). Those are the same for all three. UniMate is loaded and warm, as on a server.

*Credits: UniMate (Mou et al., SIGGRAPH Asia 2026; code MIT, weights CC BY-NC 4.0, which allows class use); pupper-mjlab
and its `motion_design` / `unimate` modules (CS 123); Gemini Pupper was created by Peng Xu (Google DeepMind).*

## 1. Setup
Clones pupper-mjlab and installs it, the same way the course notebooks do. Set `SAVE_TO_DRIVE` to keep results across disconnects.

In [ ]:
# @title Setup: pupper-mjlab, Drive, experiments (about 2 minutes)
REPO_URL = "https://github.com/cs123-stanford/pupper-mjlab.git"
BRANCH = "main"
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
# Keeps results and ratings in Google Drive, so a crash or disconnect loses nothing.
# If Drive won't mount, results go to /content (lost when the runtime resets) and this cell says so.

import os
import subprocess
import sys
from pathlib import Path

os.environ["MUJOCO_GL"] = "egl"  # render without a screen

gpu = subprocess.run(
  ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
  capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu or "none")
if "RTX PRO 6000" not in gpu:
  print("WARNING: not a G4 runtime. Runtime -> Change runtime type -> G4, then run again.\n"
        "UniMate's latency is only meaningful on the GPU you would serve it from.")

if not os.path.exists("/content/pupper-mjlab"):
  !git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/pupper-mjlab
%cd /content/pupper-mjlab
!pip install -q -e .
!pip install -q mediapy

# The running kernel can't see an editable install's path hooks; wire src/
# onto the path so imports work without a runtime restart (as the course notebooks do).
sys.path.insert(0, "/content/pupper-mjlab/src")
for _m in [k for k in list(sys.modules) if k == "mjlab" or k.startswith("mjlab.")]:
  del sys.modules[_m]
import mjlab.envs  # noqa: E402,F401  smoke-test

# ---------------------------------------------------------------------------- storage
# Everything lives under RESULTS_ROOT (Drive, or /content if Drive won't mount):
#   runs/<experiment>/results.pkl   one folder per experiment; never overwritten, only added to
#   runs/<experiment>/config.json   settings, LLM model and the exact prompts it ran with
#   ratings.csv                     every rating, tagged with the experiment it belongs to
#   exports/<time>/                 exported videos + index.html
#   tuning/<time>_*.csv             tuning tables
# Results you look at (section 8 on) are a merge of the experiments you pick, written to
# /content/active_results.pkl.
import datetime
import json
import pickle
import shutil
import time

Path("/content/bench").mkdir(exist_ok=True)
DRIVE_ROOT = Path("/content/drive/MyDrive/pupper_generation_benchmark")
RESULTS_ROOT = Path("/content/bench_results")
if SAVE_TO_DRIVE:
  try:
    from google.colab import drive

    drive.mount("/content/drive")
    RESULTS_ROOT = DRIVE_ROOT
  except Exception as e:  # noqa: BLE001
    print(f"WARNING: Google Drive didn't mount ({e}).\n  Results will be saved on this runtime only and are LOST"
          " if it resets. Rerun this cell to try again, or download them (last section) when done.")
RESULTS_DIR = RESULTS_ROOT  # older cells' name for it
RUNS = RESULTS_ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
RATINGS_FILE = RESULTS_ROOT / "ratings.csv"
ACTIVE_FILE = Path("/content/active_results.pkl")
RATING_HEADER = ["experiment", "time", "rater", "request_id", "rep", "slot", "option", "blind", "match", "natural", "demo"]


def _atomic_pickle(obj, path: Path) -> None:
  tmp = path.with_suffix(".tmp")  # write, then rename: a crash mid-save can't corrupt it
  with tmp.open("wb") as f:
    pickle.dump(obj, f)
    f.flush()
    os.fsync(f.fileno())
  tmp.replace(path)


def _migrate_old_layout() -> None:
  """Earlier versions kept one results.pkl at the top: move it into runs/first_benchmark."""
  old = RESULTS_ROOT / "results.pkl"
  if old.exists():
    dest = RUNS / "first_benchmark"
    if dest.exists():
      dest = RUNS / f"first_benchmark_{int(time.time())}"
    dest.mkdir()
    for name in ("results.pkl", "results.csv", "tune_hybrid.csv", "tune_text.csv", "videos", "calibration"):
      if (RESULTS_ROOT / name).exists():
        shutil.move(str(RESULTS_ROOT / name), str(dest / name))
    recs = pickle.loads((dest / "results.pkl").read_bytes())
    for r in recs:
      r.setdefault("experiment", dest.name)
    _atomic_pickle(recs, dest / "results.pkl")
    (dest / "config.json").write_text(json.dumps({"created": "before experiments", "note": "moved from the old layout"}))
    print(f"moved your earlier results into {dest} (nothing deleted)")
  if RATINGS_FILE.exists():
    lines = RATINGS_FILE.read_text().splitlines()
    if lines and not lines[0].startswith("experiment,"):
      RATINGS_FILE.write_text("\n".join(["experiment," + lines[0]] + ["first_benchmark," + ln for ln in lines[1:]]) + "\n")
      print("tagged your earlier ratings with experiment first_benchmark")


def experiments() -> list[str]:
  """Experiment names, oldest first."""
  def created(d):
    try:
      c = json.loads((d / "config.json").read_text()).get("created", "")
    except Exception:  # noqa: BLE001
      c = ""
    return c if c[:1].isdigit() else "0", d.stat().st_mtime
  return [d.name for d in sorted((d for d in RUNS.iterdir() if (d / "results.pkl").exists()), key=created)]


def load_experiment(name: str) -> list:
  path = RUNS / name / "results.pkl"
  recs = pickle.loads(path.read_bytes()) if path.exists() else []
  for r in recs:
    r.setdefault("experiment", name)
  return recs


def save_experiment(name: str, recs: list) -> None:
  (RUNS / name).mkdir(parents=True, exist_ok=True)
  _atomic_pickle(recs, RUNS / name / "results.pkl")


def merge_experiments(names: list[str]) -> list:
  """All results of these experiments; for the same request, option and repeat, the later name wins."""
  merged = {}
  for name in names:
    for r in load_experiment(name):
      merged[(r["request_id"], r["option"], r.get("rep", 0))] = r
  return list(merged.values())


def append_ratings(rows: list[list]) -> None:
  new = not RATINGS_FILE.exists()
  with RATINGS_FILE.open("a", newline="") as f:
    import csv as _csv

    w = _csv.writer(f)
    if new:
      w.writerow(RATING_HEADER)
    w.writerows(rows)


_migrate_old_layout()
print("mjlab ready; results under", RESULTS_ROOT)
_ex = experiments()
print("experiments:", ", ".join(f"{e} ({len(load_experiment(e))} results)" for e in _ex) if _ex else "none yet")

## 2. Install UniMate
The course's install script: UniMate in its own Python 3.10 environment, plus its 1.2 GB of weights from Hugging Face. The cell checks the result and stops with the real error if the install failed.

In [ ]:
# @title Install UniMate (about 3 minutes; its own Python environment + 1.2 GB of weights)
# Runs the course's install script, then checks that it worked: `!bash` doesn't
# stop the notebook when a script fails, so a failed install would otherwise
# only show up later, as "No module named numpy" when UniMate starts.
import shutil

UNIMATE_DIR = "/content/UniMate"
UNIMATE_EXP, UNIMATE_STEP = "unimate_uniml3d_f60_v3", 150000  # as in install.sh

# Colab sets UV_* variables (constraints for its own Python) that would also apply to
# UniMate's separate Python 3.10 environment; run the script without them.
_uv_vars = sorted(k for k in os.environ if k.startswith("UV_"))
_env = {k: v for k, v in os.environ.items()
        if not k.startswith("UV_") and k not in ("PYTHONPATH", "VIRTUAL_ENV")}
if _uv_vars:
  print("running without Colab's", ", ".join(_uv_vars))
_py = f"{UNIMATE_DIR}/.venv/bin/python"


def _venv_ok() -> bool:
  return Path(_py).exists() and subprocess.run(
    [_py, "-c", "import numpy, torch, transformers, huggingface_hub"], env=_env, capture_output=True,
  ).returncode == 0


# An earlier failed install can leave an environment marked done but empty
# (install.sh then skips the installs): start that one over.
if Path(f"{UNIMATE_DIR}/.venv/.done").exists() and not _venv_ok():
  print("removing an incomplete UniMate environment from an earlier install")
  shutil.rmtree(f"{UNIMATE_DIR}/.venv")
_proc = subprocess.run(
  ["bash", "src/mjlab/tasks/tracking/config/pupper/unimate/install.sh", UNIMATE_DIR],
  env=_env, capture_output=True, text=True,
)
print(_proc.stdout[-2000:])
if _proc.returncode != 0:
  print(_proc.stderr[-6000:])
  raise RuntimeError(f"install.sh failed (exit {_proc.returncode}): the error is just above")

# Check the environment and the weights.
_chk = subprocess.run(
  [_py, "-c", "import sys, numpy, torch, transformers; print(sys.version.split()[0], 'numpy', numpy.__version__,"
              " 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
  env=_env, capture_output=True, text=True,
)
if _chk.returncode != 0:
  print(_chk.stderr[-3000:])
  raise RuntimeError("UniMate's environment is incomplete. Delete it and rerun this cell:"
                     f"  !rm -rf {UNIMATE_DIR}/.venv")
print("UniMate environment:", _chk.stdout.strip())
_ckpt = Path(UNIMATE_DIR) / "outputs" / UNIMATE_EXP / "checkpoints" / f"checkpoint_step_{UNIMATE_STEP}.pt"
if not _ckpt.exists():
  raise RuntimeError(f"checkpoint missing: {_ckpt}. Rerun this cell (disk full? offline?)")
print(f"checkpoint: {_ckpt.name}, {_ckpt.stat().st_size / 1e9:.2f} GB")

## 3. The LLM
Pick the backend and model. The cell prints the models you can use and times a tiny request.

Which model is fair? In practice Gemini Live would call a motion tool, and the tool would ask a text model for the
keyframes or prompts. That model doesn't have to be the Live model: a stronger one gives better keyframes, a faster
one lower latency. **Use the same model for all three options.** You can rerun the whole benchmark with another model;
each result records which model made it.

In [ ]:
# @title LLM backend (writes keyframes and UniMate prompts)
# "colab": google.colab.ai, free with Colab (higher limits on Pro), no API key.
#          Gemini models served through Colab's proxy.
# "aistudio": the Gemini API with your AI Studio key, read from Colab Secrets
#          (key icon in the left bar -> add GEMINI_API_KEY, notebook access on).
#          The key is never printed or written to a file. Costs credits.
LLM_BACKEND = "colab"  # @param ["colab", "aistudio"]
LLM_MODEL = ""  # @param {type:"string"}
# Empty = the backend's default (Colab AI's default Gemini Flash; for AI Studio set
# one, e.g. a current "gemini-...-flash" from the list printed below).

import inspect
import time
from time import perf_counter


class LLM:
  """One text-in, text-out call; returns (text, seconds, seconds_waited_on_rate_limits)."""

  def __init__(self, backend: str, model: str = ""):
    self.backend = backend
    if backend == "colab":
      from google.colab import ai

      self._ai = ai
      try:
        self.available = ai.list_models()
      except Exception as e:  # noqa: BLE001
        self.available = [f"(could not list models: {e})"]
      default = inspect.signature(ai.generate_text).parameters["model_name"].default
      self.model = model or default
    elif backend == "aistudio":
      from google import genai
      from google.colab import userdata

      self._types = genai.types
      self._client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
      self.available = [m.name.removeprefix("models/") for m in self._client.models.list()
                        if "generateContent" in (m.supported_actions or [])]
      if not model:
        raise ValueError("Set LLM_MODEL for AI Studio, e.g. one of: " + ", ".join(
          m for m in self.available if "flash" in m)[:600])
      self.model = model
    else:
      raise ValueError(f"unknown backend {backend!r}")

  def _once(self, system: str, user: str) -> str:
    if self.backend == "colab":
      # Colab AI takes one prompt (no system role): system text goes first.
      text = self._ai.generate_text(system + "\n\n---\n\n" + user, model_name=self.model)
    else:
      resp = self._client.models.generate_content(
        model=self.model, contents=user,
        config=self._types.GenerateContentConfig(
          system_instruction=system, response_mime_type="application/json", temperature=0.4),
      )
      text = resp.text
    if not text:
      raise RuntimeError("empty reply")
    return text

  def __call__(self, system: str, user: str, retries: int = 6):
    waited = 0.0
    for attempt in range(retries):
      t0 = perf_counter()
      try:
        return self._once(system, user), perf_counter() - t0, waited
      except Exception as e:  # noqa: BLE001
        msg = str(e).lower()
        transient = any(s in msg for s in ("429", "rate", "quota", "resource", "unavailable",
                                            "503", "500", "overloaded", "timeout", "empty reply"))
        if not transient or attempt == retries - 1:
          raise
        pause = min(60.0, 4.0 * 2**attempt)
        print(f"  [llm] {type(e).__name__}: {str(e)[:120]}... retrying in {pause:.0f} s")
        time.sleep(pause)
        waited += pause
    raise RuntimeError("unreachable")


llm = LLM(LLM_BACKEND, LLM_MODEL)
print("backend:", llm.backend, "| model:", llm.model)
print("available:", llm.available)
lat = []
for _ in range(3):
  text, s, _w = llm("Reply with JSON only.", 'Reply with exactly {"ok": true}')
  lat.append(s)
print("reply:", text.strip()[:80])
print("round-trip latency for a tiny prompt (s):", [round(x, 2) for x in lat])

## 4. The pipelines
Library code in six cells. You don't need to edit them, but they are what's being compared:

- **Worker:** extends the course's UniMate worker with *in-betweening*. UniMate pins frames by resetting them at every
  sampling step (its own replacement sampler). We pin the positions of all 19 rig joints and the body's height and
  heading at each keyframe, and let UniMate fill in everything else. Motions longer than 2 s are chained in windows,
  the way UniMate's motion expansion does it.
- **Core:** the keyframe format, the interpolation, the Pupper ↔ UniMate rig map, a fast CPU physics replay (the gate),
  and the metrics.
- **UniMate client:** sampling, plus clean-up of UniMate output: shifting the body to match the keyframes, foot locking,
  and ranking samples before the slow IK fit.
- **Prompts:** what the LLM is told. **Read this one.** It is half of options 1 and 2, and the easiest place to make
  either one better. Pupper's reach numbers in it are measured with the course IK, not guessed.
- **Requests:** the 12 test requests (precise / style / sequence), and how a precise request's numbers are measured
  on the final motion. Add your own.
- **Pipelines:** the three options, and `Config` (all the settings in one place).

In [ ]:
%%writefile /content/pupper_bench_worker.py
"""Benchmark worker: the course's UniMate worker plus keyframe in-betweening.

Runs inside UniMate's own environment (started by the notebook, never by hand).
It imports pupper-mjlab's worker.py for everything that already works there:
loading the model, describing Pupper's skeleton to UniMate, encoding text. On
top of that it adds one command, "sample", that does all three generation modes
with one sampler:

  * free text-to-motion (option 3): no pinned frames, optionally chained prompts;
  * keyframe in-betweening (option 2): some frames' poses are pinned during
    sampling and UniMate fills in the motion around them;
  * longer than 2 s: windows of 60 frames chained with a 10-frame overlap, the
    way UniMate's own motion expansion does it.

Pinning is UniMate's own replacement sampler (unimate/inference/
motion_inbetweening.py): at every Euler step the pinned entries are reset to
(1 - t) * noise + t * target. UniMate pins whole frames of a real motion; we only
know *positions* (from keyframes), so we pin the position channels (RIFKE) of
every joint plus the root's height and facing, and let UniMate fill the rotation
and velocity channels.

Protocol (one JSON line in, one out), on top of worker.py's register/generate:
  {"cmd": "sample", "skeleton": "pupper", "prompts": [...], "n": 4, "seed": 0,
   "cfg": 3.0, "steps": 50, "solver": "euler" | "dopri5", "frames": N,
   "dense": [[[x, y, z] x 19] x N] | null,  # target poses, robot frame, lowest joint at z=0
   "dense_yaw": [N radians], "pin": [frame, ...], "pin_facing": true,
   "release_t": 1.0, "decode": "ric" | "fk", "overlap": 10, "amp": false,
   "out": "path.npz"}
  {"cmd": "selftest", "skeleton": "pupper", "prompt": ..., "pelvis": 0, "chest": 2}
"""

from __future__ import annotations

import json
import math
import os
import sys
import time

sys.path.insert(0, os.environ["PUPPER_UNIMATE_PY"])  # pupper-mjlab's unimate/ folder
import worker as W  # noqa: E402  (parses --unimate/--exp/--step, imports UniMate)

import numpy as np  # noqa: E402
import torch  # noqa: E402
from Quaternions import Quaternions  # noqa: E402
from unimate.inference.generate import ClassifierFreeSampleModel  # noqa: E402
from unimate.inference.motion_expansion import (  # noqa: E402
  first_frame_facing,
  rotate_segment_facing,
)
from unimate.utils.motion_utils import compute_rifke  # noqa: E402
from unimate.utils.rotation_conversions import (  # noqa: E402
  matrix_to_quaternion_np,
  rotation_6d_to_matrix_np,
)

PROTO = W.PROTO  # worker.py moved print() to stderr; the protocol goes here
CONST_STD = 1e-6  # channels with a smaller std are constant (motion_expansion.py)


class BenchWorker(W.Worker):
  # ---- helpers ---------------------------------------------------------------
  def _conds(self, sk, prompts, n):
    conds = []
    for text in prompts:
      enc = self.encode(text)
      _motion, cond = W.mixture_batch_collate([self._batch(sk, enc) for _ in range(n)])
      conds.append(
        {k: v.to(self.device) if torch.is_tensor(v) else v for k, v in cond.items()}
      )
    return conds

  def _stats(self, cond, J):
    mean = cond["mean"][0, :J].double().cpu().numpy()  # (J, D)
    std = cond["std"][0, :J].double().cpu().numpy()
    return mean, std

  def encode_frames(self, sk, pos, yaw, cond):
    """Target poses -> normalized UniMate features with the pinnable channels filled.

    pos (K, J, 3): joint positions in the registered order, robot frame
    (x forward, y left, z up, metres) with the ground at z = 0 for joint centres
    (the frame worker.py decodes into). yaw (K,): heading in radians.
    Returns (K, J, D); only the RIFKE positions (all joints) and the root's
    facing are meaningful, everything else is left at 0 (= the mean).
    """
    J = len(sk["names"])
    p = W.to_raw(np.asarray(pos, float)) * sk["scale"]  # canonical: Y up, Z forward
    p = p[:, sk["bfs_order"]]
    K = len(p)
    yaw = np.asarray(yaw, float)
    fwd = np.stack([np.sin(yaw), np.zeros(K), np.cos(yaw)], -1)  # robot +x heading in raw axes
    q = Quaternions.between(fwd, np.tile([0.0, 0.0, 1.0], (K, 1)))  # facing -> +Z
    feats = np.zeros((K, J, self.config.dataset.feature_len))
    feats[..., 0:3] = compute_rifke(p, q)
    feats[:, 0, 3:9] = q.rotation_matrix(cont6d=True)
    mean, std = self._stats(cond, J)
    return np.where(
      std[None] < CONST_STD, 0.0, (feats - mean[None]) / np.maximum(std[None], CONST_STD)
    )

  def _ode(self, cond, shape, cfg, steps, x1=None, hard=None, soft=None,
           release_t=1.0, solver="euler"):
    """Flow-matching sampler with optional replacement pinning.

    hard: entries pinned for the whole integration (chain overlaps).
    soft: entries pinned while t <= release_t (keyframes); release_t < 1 lets
    the last steps run free so the motion can relax around the keyframes.
    """
    if x1 is None and solver == "dopri5":  # UniMate's default text-to-motion path
      return W.generate_samples(
        model=self.model, cond=cond, motion_shape=shape, diff_model="flow",
        diffusion=self.transport, gen_diffusion=self.sampler, device=self.device,
        cfg_scale=cfg,
      )
    net = ClassifierFreeSampleModel(self.model, cfg_scale=cfg) if cfg > 1.0 else self.model
    t0, t1 = self.transport.check_interval(
      self.transport.train_eps, self.transport.sample_eps,
      sde=False, eval=True, reverse=False, last_step_size=0.0,
    )
    t0, t1 = float(t0), float(t1)
    eps = torch.randn(shape, device=self.device)
    x = eps.clone()

    def pin(x, t):
      if x1 is None:
        return x
      m = hard
      if soft is not None and t <= release_t + 1e-9:
        m = soft if m is None else (m | soft)
      if m is None:
        return x
      return torch.where(m, (1.0 - t) * eps + t * x1, x)

    x = pin(x, t0)
    ts = np.linspace(t0, t1, steps + 1)
    for i in range(steps):
      tb = torch.full((shape[0],), float(ts[i]), device=self.device, dtype=x.dtype)
      x = x + float(ts[i + 1] - ts[i]) * net(x, tb, cond=cond)
      x = pin(x, float(ts[i + 1]))
    return x

  # ---- commands ----------------------------------------------------------------
  def sample(self, req):
    sk = self.skeletons[req["skeleton"]]
    n = int(req.get("n", 4))
    # TF32 matmuls: off by default in PyTorch (full fp32); several times faster on A100/Blackwell.
    torch.backends.cuda.matmul.allow_tf32 = bool(req.get("tf32", False))
    if req.get("seed") is not None:
      torch.manual_seed(int(req["seed"]))
      np.random.seed(int(req["seed"]))
    dcfg = self.config.dataset
    T = dcfg.max_motion_length
    overlap = int(req.get("overlap", 10))
    frames = int(req.get("frames", T))
    S = 1 if frames <= T else 1 + math.ceil((frames - T) / (T - overlap))
    prompts = [p for p in req["prompts"] if p.strip()]
    seg_prompts = [prompts[min(i, len(prompts) - 1)] for i in range(S)]
    solver = req.get("solver", "euler")
    steps = int(req.get("steps", 50))
    cfg = float(req.get("cfg", self.config.sampling.cfg_scale))
    release_t = float(req.get("release_t", 1.0))
    J = len(sk["names"])
    D = dcfg.feature_len
    shape = (n, dcfg.max_joints, D, T)

    dense = req.get("dense")
    pins = sorted({int(f) for f in req.get("pin", [])})
    if dense is not None:
      dense = np.asarray(dense, float)
      dyaw = np.asarray(req["dense_yaw"], float)

    t_start = time.time()
    conds = self._conds(sk, seg_prompts, n)
    if self.device.type == "cuda":
      torch.cuda.synchronize()
    enc_s = time.time() - t_start

    t0 = time.time()
    segs, prev = [], None
    amp = bool(req.get("amp", False)) and self.device.type == "cuda"
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
      for s in range(S):
        cond = conds[s]
        start = s * (T - overlap)
        local = []
        if dense is not None:
          local = [
            f - start for f in pins
            if start <= f < start + T and f < len(dense) and (s == 0 or f - start >= overlap)
          ]
          # A window can't see keyframes in the next one; the next window starts
          # from this one's tail. When a pinned frame comes soon after the seam
          # (within ~0.5 s), pin the window's last frame to the keyframe path (the
          # guide) so the hand-over already heads for it; otherwise that keyframe
          # can be unreachable. Not otherwise: between keyframes the guide is only
          # an interpolation (with just a start and end stand, it's "stand still"),
          # and pinning it there would freeze the motion mid-way.
          end = start + T - 1
          soon = any(end < f <= end + 15 for f in pins)
          if s < S - 1 and soon and end < len(dense) and (T - 1) not in local:
            local.append(T - 1)
          local.sort()
        x1 = hard = soft = None
        if local:
          idx = [start + f for f in local]
          feats = self.encode_frames(sk, dense[idx], dyaw[idx], cond)  # (K, J, D)
          x1 = torch.zeros(shape, device=self.device)
          x1[:, :J, :, local] = torch.from_numpy(feats.transpose(1, 2, 0)).float().to(self.device)[None]
          soft = torch.zeros(shape, dtype=torch.bool, device=self.device)
          soft[:, :J, 0:3, local] = True
          if req.get("pin_facing", True):
            soft[:, 0, 3:9, local] = True
        q_seed = None
        if prev is not None:
          # Same seam handling as UniMate's expand_motion_chain: work in the frame
          # where the overlap's first frame faces +Z, rotate back afterwards.
          tail = prev[..., -overlap:]
          q_seed = first_frame_facing(tail, cond)
          tail = rotate_segment_facing(tail, cond, q_seed)
          if x1 is None:
            x1 = torch.zeros(shape, device=self.device)
          else:
            x1 = rotate_segment_facing(x1, cond, q_seed)
          x1[..., :overlap] = tail
          hard = torch.zeros(shape, dtype=torch.bool, device=self.device)
          hard[..., :overlap] = True
        seg = self._ode(cond, shape, cfg, steps, x1=x1, hard=hard, soft=soft,
                        release_t=release_t, solver=solver)
        seg = seg.float()
        if q_seed is not None:
          seg = rotate_segment_facing(seg, cond, [-q for q in q_seed])
        segs.append(seg)
        prev = seg
    if self.device.type == "cuda":
      torch.cuda.synchronize()
    gen_s = time.time() - t0

    t1 = time.time()
    chain = torch.cat([segs[0]] + [g[..., overlap:] for g in segs[1:]], dim=-1)
    mean, std = self._stats(conds[0], J)
    decode = req.get("decode", "ric")
    out = []
    for i in range(n):
      m = chain[i, :J].double().cpu().permute(2, 0, 1).numpy() * std[None] + mean[None]
      if decode == "ric":
        pos = W.recover_unimate_joint_pos_from_ric(m)
      else:
        pos = W.recover_unimate_joint_pos_from_rot(m, sk["parents"], sk["offsets"])
      out.append(pos)
    pos = W.from_raw(np.stack(out) / sk["scale"])[:, :, np.argsort(sk["bfs_order"])]
    seams = [T + k * (T - overlap) for k in range(S - 1)]
    if seams and dense is None:
      # Text-only chains: worker.py's seam smoothing. Not with keyframes: the
      # "jump" it removes can be a pinned keyframe, and the seams are pinned anyway.
      pos = W.smooth_seams(pos, seams)
    np.savez(req["out"], positions=pos, fps=30.0, seams=np.array(seams))
    return {
      "ok": True, "out": req["out"], "seconds": round(gen_s, 4),
      "encode_seconds": round(enc_s, 4), "decode_seconds": round(time.time() - t1, 4),
      "segments": S, "shape": list(pos.shape),
    }

  def selftest(self, req):
    """Checks that encode_frames speaks UniMate's representation.

    Generates one free sample, decodes it to robot-frame positions the way
    "sample" does, re-encodes those positions, and compares with what the model
    produced: RIFKE positions should match to float precision (axes, scale,
    joint order and ground are right), and the facing re-derived from the body
    (pelvis -> chest) should be close to the model's own facing channel (the
    facing convention is right).
    """
    sk = self.skeletons[req["skeleton"]]
    torch.manual_seed(int(req.get("seed", 0)))
    dcfg = self.config.dataset
    J = len(sk["names"])
    cond = self._conds(sk, [req.get("prompt", "An object turns around to the left.")], 1)[0]
    shape = (1, dcfg.max_joints, dcfg.feature_len, dcfg.max_motion_length)
    with torch.no_grad():
      x = self._ode(cond, shape, 3.0, 50)
    mean, std = self._stats(cond, J)
    m = x[0, :J].double().cpu().permute(2, 0, 1).numpy() * std[None] + mean[None]
    pos_c = W.recover_unimate_joint_pos_from_ric(m)
    robot = W.from_raw(pos_c / sk["scale"])[:, np.argsort(sk["bfs_order"])]
    # (a) RIFKE round trip, using the model's own facing
    qa = Quaternions(matrix_to_quaternion_np(rotation_6d_to_matrix_np(m[:, 0, 3:9])))
    fwd_raw = (-qa) * np.tile([0.0, 0.0, 1.0], (len(m), 1))
    yaw_model = np.arctan2(fwd_raw[:, 0], fwd_raw[:, 2])  # raw X = robot y, raw Z = robot x
    enc = self.encode_frames(sk, robot, yaw_model, cond) * np.maximum(std[None], CONST_STD) + mean[None]
    rif_err = float(np.abs(enc[..., 0:3] - m[..., 0:3]).max() / sk["scale"])
    # (b) facing from the body vs the model's facing
    fwd = robot[:, req["chest"], :2] - robot[:, req["pelvis"], :2]
    yaw_body = np.arctan2(fwd[:, 1], fwd[:, 0])
    enc_b = self.encode_frames(sk, robot, yaw_body, cond) * np.maximum(std[None], CONST_STD) + mean[None]
    qb = Quaternions(matrix_to_quaternion_np(rotation_6d_to_matrix_np(enc_b[:, 0, 3:9])))
    dots = np.clip(np.abs((qa.qs * qb.qs).sum(-1)), 0.0, 1.0)
    ang = np.degrees(2 * np.arccos(dots))
    six = float(np.abs(qa.rotation_matrix(cont6d=True) - m[:, 0, 3:9]).max())
    feet_z = robot[..., 2]
    return {
      "ok": True,
      "rifke_roundtrip_err_m": rif_err,
      "facing_body_vs_model_deg_median": float(np.median(ang)),
      "facing_body_vs_model_deg_max": float(ang.max()),
      "yaw_turned_deg": float(np.degrees(np.unwrap(yaw_model)[-1] - yaw_model[0])),
      "sixd_roundtrip_err": six,
      "lowest_joint_z_m": float(feet_z.min()),
      "scale": float(sk["scale"]),
    }


def main():
  w = BenchWorker()
  print(json.dumps({"ok": True, "ready": True}), file=PROTO, flush=True)
  for line in sys.stdin:
    line = line.strip()
    if not line:
      continue
    try:
      req = json.loads(line)
      res = getattr(w, req["cmd"])(req)
    except Exception as e:  # noqa: BLE001
      import traceback

      traceback.print_exc(file=sys.stderr)
      res = {"ok": False, "error": f"{type(e).__name__}: {e}"}
    print(json.dumps(res), file=PROTO, flush=True)


if __name__ == "__main__":
  main()

In [ ]:
# @title Core: keyframe format, interpolation, rig map, physics gate, metrics
# Everything here is plain CPU code on top of pupper-mjlab's motion_design.py
# (md) and unimate/__init__.py (U). All three options share it, so they are
# checked and scored by exactly the same code.
import dataclasses
import json
import math
import re
import time
import zlib
from pathlib import Path

import mujoco
import numpy as np
from scipy.spatial import ConvexHull, QhullError

from mjlab.asset_zoo.robots.pupper_v3.pupper_constants import DEFAULT_POSE, JOINT_NAMES
from mjlab.tasks.tracking.config.pupper import motion_design as md
from mjlab.tasks.tracking.config.pupper import unimate as U
from mjlab.tasks.tracking.config.pupper.env_cfgs import get_spec_with_hulls

FPS = 50.0  # Pupper motions
GEN_FPS = U.GEN_FPS  # UniMate, 30 fps
LEGS = md.LEGS
H = md.STAND_HEIGHT
KIN = md.kinematics()
STAND_FEET = {leg: np.array(p, float) for leg, p in md.stand_feet().items()}


# ============================================================================ rotations / motions
def quat_to_rpy(q) -> np.ndarray:
  """(w, x, y, z) -> (roll, pitch, yaw) degrees; inverse of md.rpy_to_quat (z-y-x)."""
  R = md.quat_to_mat(q)
  yaw = math.atan2(R[1, 0], R[0, 0])
  pitch = math.asin(max(-1.0, min(1.0, -R[2, 0])))
  roll = math.atan2(R[2, 1], R[2, 2])
  return np.degrees([roll, pitch, yaw])


def rpy_series(motion) -> np.ndarray:
  rpy = np.array([quat_to_rpy(q) for q in motion.root_quat])
  rpy[:, 2] = np.degrees(np.unwrap(np.radians(rpy[:, 2])))
  return rpy


def quat_angle_deg(a, b) -> float:
  a, b = np.asarray(a, float), np.asarray(b, float)
  dot = abs(float(a @ b)) / (np.linalg.norm(a) * np.linalg.norm(b))
  return math.degrees(2 * math.acos(min(1.0, dot)))


def make_motion(root_pos, root_quat, joint_pos) -> md.Motion:
  m = md.Motion(
    fps=FPS,
    root_pos=np.asarray(root_pos, float),
    root_quat=np.asarray(root_quat, float),
    joint_pos=np.asarray(joint_pos, float),
  )
  m.report = md.check_motion(m)
  return m


def concat_motions(a: md.Motion, b: md.Motion) -> md.Motion:
  """a then b; b's first frame is dropped (it repeats a's last one)."""
  return make_motion(
    np.concatenate([a.root_pos, b.root_pos[1:]]),
    np.concatenate([a.root_quat, b.root_quat[1:]]),
    np.concatenate([a.joint_pos, b.joint_pos[1:]]),
  )


def stretch_motion(motion: md.Motion, factor: float) -> md.Motion:
  """Play the motion `factor` times slower (same poses, more frames)."""
  n = len(motion.root_pos)
  m = int(round((n - 1) * factor)) + 1
  f = np.linspace(0.0, n - 1, m)
  i0 = np.floor(f).astype(int)
  i1 = np.minimum(i0 + 1, n - 1)
  a = (f - i0)[:, None]
  q0, q1 = motion.root_quat[i0], motion.root_quat[i1].copy()
  q1[(q0 * q1).sum(1) < 0] *= -1
  q = (1 - a) * q0 + a * q1
  q /= np.linalg.norm(q, axis=1, keepdims=True)
  return make_motion(
    (1 - a) * motion.root_pos[i0] + a * motion.root_pos[i1],
    q,
    (1 - a) * motion.joint_pos[i0] + a * motion.joint_pos[i1],
  )


def feet_series(motion) -> np.ndarray:
  """World foot centres (T, 4, 3) in LEGS order."""
  out = np.zeros((len(motion.root_pos), 4, 3))
  for i in range(len(out)):
    f = KIN.feet(motion.root_pos[i], motion.root_quat[i], motion.joint_pos[i])
    out[i] = [f[leg] for leg in LEGS]
  return out


def motion_to_dict(m: md.Motion) -> dict:
  return {"root_pos": m.root_pos, "root_quat": m.root_quat, "joint_pos": m.joint_pos}


def motion_from_dict(d: dict) -> md.Motion:
  return make_motion(d["root_pos"], d["root_quat"], d["joint_pos"])


# ============================================================================ the keyframe format
# What Gemini writes (one JSON object). Everything is relative to the stand, so
# the numbers are small and readable:
#   {"t": 0.0, "pose": "stand"}
#   {"t": 0.6, "body": {"x": 0, "y": 0, "dz": -0.02, "roll": 0, "pitch": 10, "yaw": 0},
#    "feet": {"front_l": [0.0, 0.0, 0.04]}, "ease": "stop", "hold": 0.5}
# body: x, y = body position (m) from the start; dz = height change from standing;
# roll/pitch/yaw in degrees. feet: [dx, dy, dz] (m, world frame) from where that
# foot stands; feet left out stay planted. ease: "stop" comes to rest at the
# keyframe (default), "flow" passes through it. hold: seconds to stay there.
SCHEMA_LIMITS = dict(  # generous sanity bounds; feasibility is the verifier's job
  t_max=10.0, min_gap=0.08, body_xy=0.10, dz=(-0.13, 0.06),
  roll=35.0, pitch=50.0, yaw=60.0, foot_xy=0.10, foot_dz=(0.0, 0.12),
)


@dataclasses.dataclass
class KF:
  t: float
  body: np.ndarray  # x, y, dz, roll, pitch, yaw
  feet: dict  # leg -> np.array([dx, dy, dz])
  ease: str = "stop"
  src: int = -1  # index in Gemini's list (for error messages)

  def md_keyframe(self) -> md.Keyframe:
    x, y, dz, r, p, yw = (float(v) for v in self.body)
    feet = {}
    for leg in LEGS:
      off = self.feet.get(leg, np.zeros(3))
      s = STAND_FEET[leg]
      feet[leg] = (s[0] + off[0], s[1] + off[1], md.FOOT_RADIUS + off[2])
    return md.Keyframe(self.t, (x, y, H + dz), (r, p, yw), feet=feet)

  def channels(self) -> np.ndarray:
    """18 numbers interpolated between keyframes: root xyz, rpy, 4 feet xyz."""
    k = self.md_keyframe()
    return np.concatenate([k.root_pos, k.root_rpy] + [np.array(k.feet[leg]) for leg in LEGS])


def _num(v, what, errs):
  try:
    x = float(v)
  except (TypeError, ValueError):
    errs.append(f"{what} must be a number, got {v!r}")
    return 0.0
  if not math.isfinite(x):
    errs.append(f"{what} must be finite")
    return 0.0
  return x


def parse_keyframes(items) -> tuple[list[KF], list[str]]:
  """Gemini's keyframe list -> KFs (holds expanded), plus every problem found."""
  L = SCHEMA_LIMITS
  errs: list[str] = []
  if not isinstance(items, list) or len(items) < 2:
    return [], ["'keyframes' must be a list of at least two keyframes (start and end standing)"]
  kfs: list[KF] = []
  for i, it in enumerate(items):
    where = f"keyframe {i}"
    if not isinstance(it, dict):
      errs.append(f"{where} must be an object")
      continue
    t = _num(it.get("t"), f"{where}: t", errs)
    where = f"keyframe {i} (t={t:.2f})"
    body = np.zeros(6)
    feet = {}
    if it.get("pose") not in (None, "stand"):
      errs.append(f"{where}: pose can only be \"stand\"")
    if it.get("pose") != "stand":
      b = it.get("body", {}) or {}
      if not isinstance(b, dict):
        errs.append(f"{where}: body must be an object")
        b = {}
      for j, key in enumerate(("x", "y", "dz", "roll", "pitch", "yaw")):
        body[j] = _num(b.get(key, 0.0), f"{where}: body.{key}", errs)
      for key in b:
        if key not in ("x", "y", "dz", "roll", "pitch", "yaw"):
          errs.append(f"{where}: unknown body field {key!r} (use x, y, dz, roll, pitch, yaw)")
      f = it.get("feet", {}) or {}
      if not isinstance(f, dict):
        errs.append(f"{where}: feet must be an object")
        f = {}
      for leg, off in f.items():
        if leg not in LEGS:
          errs.append(f"{where}: unknown foot {leg!r}; feet are {list(LEGS)}")
          continue
        if not isinstance(off, (list, tuple)) or len(off) != 3:
          errs.append(f"{where}: feet.{leg} must be [dx, dy, dz]")
          continue
        feet[leg] = np.array([_num(v, f"{where}: feet.{leg}", errs) for v in off])
    ease = it.get("ease", "stop")
    if ease not in ("stop", "flow"):
      errs.append(f"{where}: ease must be \"stop\" or \"flow\"")
      ease = "stop"
    # range checks
    if abs(body[0]) > L["body_xy"] or abs(body[1]) > L["body_xy"]:
      errs.append(f"{where}: body x/y must stay within +-{L['body_xy']} m of the start")
    if not L["dz"][0] <= body[2] <= L["dz"][1]:
      errs.append(f"{where}: body dz must be in [{L['dz'][0]}, {L['dz'][1]}] m")
    for j, key in ((3, "roll"), (4, "pitch"), (5, "yaw")):
      if abs(body[j]) > L[key]:
        errs.append(f"{where}: {key} must be within +-{L[key]} deg")
    for leg, off in feet.items():
      if abs(off[0]) > L["foot_xy"] or abs(off[1]) > L["foot_xy"]:
        errs.append(f"{where}: feet.{leg} dx/dy must be within +-{L['foot_xy']} m")
      if not L["foot_dz"][0] <= off[2] <= L["foot_dz"][1]:
        errs.append(f"{where}: feet.{leg} dz must be in [0, {L['foot_dz'][1]}] m (0 = on the ground)")
    kfs.append(KF(t, body, feet, ease, i))
    hold = it.get("hold")
    if hold is not None:
      h = _num(hold, f"{where}: hold", errs)
      if h < 0:
        errs.append(f"{where}: hold must be >= 0")
      elif h > 0:
        kfs[-1].ease = "stop"
        kfs.append(KF(t + h, body.copy(), {k: v.copy() for k, v in feet.items()}, "stop", i))
  if errs:
    return kfs, errs
  for a, b in zip(kfs, kfs[1:]):
    if b.t - a.t < L["min_gap"] - 1e-9:
      errs.append(
        f"keyframe {b.src} (t={b.t:.2f}) comes {b.t - a.t:.2f} s after the previous one"
        f" (hold included); keyframes must be at least {L['min_gap']} s apart and in time order"
      )
  if abs(kfs[0].t) > 1e-9:
    errs.append("the first keyframe must be at t = 0")
  if kfs[-1].t > L["t_max"]:
    errs.append(f"the motion lasts {kfs[-1].t:.1f} s; keep it under {L['t_max']} s")
  for kf, name in ((kfs[0], "first"), (kfs[-1], "last")):
    if np.abs(kf.body[2:]).max() > 1e-6 or np.abs(kf.body[:2]).max() > 0.005 or any(
      np.abs(o).max() > 1e-6 for o in kf.feet.values()
    ):
      errs.append(f"the {name} keyframe must be {{\"pose\": \"stand\"}}: start and end standing")
  return kfs, errs


# ============================================================================ interpolation
def _quintic(s):
  """Quintic Hermite basis with zero acceleration at both ends."""
  s2, s3 = s * s, s * s * s
  s4, s5 = s3 * s, s3 * s2
  return (
    1 - 10 * s3 + 15 * s4 - 6 * s5,  # p0
    s - 6 * s3 + 8 * s4 - 3 * s5,  # v0 * h
    10 * s3 - 15 * s4 + 6 * s5,  # p1
    -4 * s3 + 7 * s4 - 3 * s5,  # v1 * h
  )


def keyframe_track(kfs: list[KF], fps: float = FPS) -> tuple[np.ndarray, np.ndarray]:
  """Frame times and the 18 interpolated channels (T, 18).

  "stop" keyframes get zero velocity (exactly the minimum-jerk curve that
  md.build_motion uses); "flow" keyframes get a monotone Catmull-Rom velocity,
  so the motion passes through them without stopping and never overshoots.
  """
  t = np.array([k.t for k in kfs])
  V = np.stack([k.channels() for k in kfs])
  M = np.zeros_like(V)
  for i in range(1, len(kfs) - 1):
    if kfs[i].ease != "flow":
      continue
    d0 = (V[i] - V[i - 1]) / (t[i] - t[i - 1])
    d1 = (V[i + 1] - V[i]) / (t[i + 1] - t[i])
    m = 0.5 * (d0 + d1)
    m = np.where(d0 * d1 <= 0, 0.0, m)  # a local extremum: stop in that channel
    lim = 3.0 * np.minimum(np.abs(d0), np.abs(d1))
    M[i] = np.clip(m, -lim, lim)
  n = int(round(t[-1] * fps)) + 1
  times = np.arange(n) / fps
  out = np.zeros((n, V.shape[1]))
  for i, ti in enumerate(times):
    j = min(int(np.searchsorted(t, ti, side="right")) - 1, len(kfs) - 2)
    h = t[j + 1] - t[j]
    s = float(np.clip((ti - t[j]) / h, 0.0, 1.0))
    b0, b1, b2, b3 = _quintic(s)
    out[i] = b0 * V[j] + b1 * h * M[j] + b2 * V[j + 1] + b3 * h * M[j + 1]
  out[:, 8::3] = np.maximum(out[:, 8::3], md.FOOT_RADIUS)  # feet z never below the ground
  return times, out


def build_from_keyframes(kfs: list[KF], interp: str = "spline") -> md.Motion:
  """Keyframes -> a 50 Hz Pupper motion with IK (md.build_motion does the IK).

  interp="minjerk": md.build_motion on the keyframes, as in the Tricks notebook
  (the robot stops at every keyframe). interp="spline": the track above, sampled
  at every frame and handed to md.build_motion frame by frame, so "flow"
  keyframes really flow.
  """
  if interp == "minjerk":
    return md.build_motion([k.md_keyframe() for k in kfs], fps=FPS)
  times, track = keyframe_track(kfs)
  frames = [
    md.Keyframe(
      float(ti), tuple(c[0:3]), tuple(c[3:6]),
      feet={leg: tuple(c[6 + 3 * i: 9 + 3 * i]) for i, leg in enumerate(LEGS)},
    )
    for ti, c in zip(times, track)
  ]
  return md.build_motion(frames, fps=FPS)


def keyframe_errors(motion: md.Motion, kfs: list[KF]) -> list[dict]:
  """How far the built (or generated) motion is from each keyframe."""
  out = []
  for kf in kfs:
    i = min(int(round(kf.t * motion.fps)), len(motion.root_pos) - 1)
    k = kf.md_keyframe()
    got = KIN.feet(motion.root_pos[i], motion.root_quat[i], motion.joint_pos[i])
    foot = {leg: float(np.linalg.norm(got[leg] - np.array(k.feet[leg]))) for leg in LEGS}
    out.append(dict(
      t=kf.t, src=kf.src, foot_err=foot,
      root_err=float(np.linalg.norm(motion.root_pos[i] - np.array(k.root_pos))),
      rot_err=quat_angle_deg(motion.root_quat[i], md.rpy_to_quat(k.root_rpy)),
    ))
  return out


# ============================================================================ rig map (Pupper <-> UniMate skeleton)
SK = U.SKELETON
RIG_IX = {n: i for i, n in enumerate(SK["names"])}
RIG_FEET = [RIG_IX[f] for _, f in SK["legs"].values()]
RIG_ROOTS = [RIG_IX[r] for r, _ in SK["legs"].values()]
RIG_LEG_ORDER = list(SK["legs"].keys())


class RigMap:
  """Pupper pose -> the 19 joints of the UniMate rig that U.fit_pupper fits to.

  Each rig joint is fixed to one Pupper body (torso, upper leg, lower leg) at
  its offset at the default stand, so the default stand reproduces U.SKELETON's
  rest pose exactly and bone lengths never change.
  """

  def __init__(self):
    m = KIN.model
    body_of_joint = {
      n: int(m.jnt_bodyid[mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_JOINT, n)])
      for n in JOINT_NAMES
    }
    names = SK["names"]
    bodies = {}
    for leg, (root, foot) in SK["legs"].items():
      knee = names[SK["parents"][names.index(foot)]]
      bodies[root] = body_of_joint[f"leg_{leg}_1"]  # hip: moves with abduction
      bodies[knee] = body_of_joint[f"leg_{leg}_2"]  # knee: end of the upper leg
      bodies[foot] = body_of_joint[f"leg_{leg}_3"]  # foot: end of the lower leg
    self.body = np.array([bodies.get(n, KIN.base_body) for n in names])
    KIN.set_state(np.array([0.0, 0.0, H]), np.array([1.0, 0, 0, 0]), np.array(DEFAULT_POSE))
    d = KIN.data
    rest = np.array(SK["rest"])
    self.off = np.stack(
      [d.xmat[b].reshape(3, 3).T @ (rest[i] - d.xpos[b]) for i, b in enumerate(self.body)]
    )

  def pose(self, root_pos, root_quat, joints) -> np.ndarray:
    KIN.set_state(root_pos, root_quat, joints)
    d = KIN.data
    R = d.xmat[self.body].reshape(-1, 3, 3)
    return d.xpos[self.body] + np.einsum("jab,jb->ja", R, self.off)

  def motion(self, motion: md.Motion) -> np.ndarray:
    return np.stack([self.pose(*fr) for fr in zip(motion.root_pos, motion.root_quat, motion.joint_pos)])


RIG = RigMap()


def resample(x: np.ndarray, src_fps: float, dst_fps: float, n_out: int | None = None) -> np.ndarray:
  """Linear resampling along axis 0."""
  dur = (len(x) - 1) / src_fps
  n_out = n_out or int(round(dur * dst_fps)) + 1
  f = np.clip(np.arange(n_out) / dst_fps * src_fps, 0, len(x) - 1)
  i0 = np.floor(f).astype(int)
  i1 = np.minimum(i0 + 1, len(x) - 1)
  a = (f - i0).reshape(-1, *([1] * (x.ndim - 1)))
  return (1 - a) * x[i0] + a * x[i1]


def unimate_guide(motion: md.Motion) -> tuple[np.ndarray, np.ndarray]:
  """A Pupper motion as UniMate targets: rig joints (N, 19, 3) at 30 fps in the
  frame UniMate's worker uses (ground at z = 0 for joint centres), and yaw (rad)."""
  rig = RIG.motion(motion)
  pos = resample(rig, FPS, GEN_FPS)
  pos[..., 2] -= U.FOOT_Z
  yaw = np.radians(rpy_series(motion)[:, 2])
  yaw = resample(yaw, FPS, GEN_FPS, len(pos))
  return pos, yaw


# ============================================================================ physics gate (fast, CPU)
class PhysicsGate:
  """Open-loop physics replay on the CPU, as play_motion.py does it on the GPU.

  Same model (hull collisions), the MJX-era solver options from
  pupper_constants (4 ms Euler steps, pyramidal cones, 2 iterations), PD
  position servos with play_motion's replay gains (kp 7.5, kd 0.25, 3 Nm),
  targets = the reference's next frame, same verdict thresholds. It skips
  mjlab's sensor/actuator latency model, so it is a little optimistic; section 5
  compares it with play_motion itself. About 20-80 ms per motion.
  """

  KP, KD, EFFORT = 7.5, 0.25, 3.0
  WOBBLE_POS_M, WOBBLE_TILT_DEG = 0.03, 10.0
  FALL_POS_M, FALL_TILT_DEG = 0.10, 45.0

  def __init__(self):
    spec = get_spec_with_hulls()
    o = spec.option
    o.timestep = 0.004
    o.integrator = mujoco.mjtIntegrator.mjINT_EULER
    o.cone = mujoco.mjtCone.mjCONE_PYRAMIDAL
    o.impratio = 10.0
    o.iterations = 2
    o.ls_iterations = 5
    o.disableflags |= mujoco.mjtDisableBit.mjDSBL_EULERDAMP
    for g in spec.geoms:
      if g.name.endswith("_3_collision"):
        g.priority = 1  # foot friction wins, as in PUPPER_COLLISION
    spec.worldbody.add_geom(
      name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[5, 5, 0.1],
      contype=1, conaffinity=1, condim=3, friction=[0.8, 0.02, 0.01],
    )
    for jn in JOINT_NAMES:
      a = spec.add_actuator(name=f"{jn}_pd", target=jn, trntype=mujoco.mjtTrn.mjTRN_JOINT)
      a.gaintype = mujoco.mjtGain.mjGAIN_FIXED
      a.biastype = mujoco.mjtBias.mjBIAS_AFFINE
      a.gainprm = [self.KP] + [0.0] * 9
      a.biasprm = [0.0, -self.KP, -self.KD] + [0.0] * 7
      a.forcelimited = mujoco.mjtLimited.mjLIMITED_TRUE
      a.forcerange = [-self.EFFORT, self.EFFORT]
    self.model = spec.compile()
    self.data = mujoco.MjData(self.model)
    m = self.model
    self.qadr = np.array(
      [m.jnt_qposadr[mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_JOINT, n)] for n in JOINT_NAMES]
    )
    self.substeps = int(round((1.0 / FPS) / m.opt.timestep))

  def replay(self, motion: md.Motion) -> dict:
    m, d = self.model, self.data
    mujoco.mj_resetData(m, d)
    d.qpos[:3] = motion.root_pos[0] + np.array([0.0, 0.0, 0.001])
    d.qpos[3:7] = motion.root_quat[0]
    d.qpos[self.qadr] = motion.joint_pos[0]
    d.ctrl[:] = motion.joint_pos[0]
    mujoco.mj_forward(m, d)
    T = len(motion.root_pos)
    traj = np.zeros((T, 19))
    traj[0] = np.concatenate([d.qpos[:7], d.qpos[self.qadr]])
    worst_pos = worst_tilt = 0.0
    fell_at = wobble_at = None
    for i in range(T - 1):
      d.ctrl[:] = motion.joint_pos[i + 1]
      for _ in range(self.substeps):
        mujoco.mj_step(m, d)
      traj[i + 1] = np.concatenate([d.qpos[:7], d.qpos[self.qadr]])
      if not np.isfinite(d.qpos).all():
        fell_at = fell_at or (i + 1) / FPS
        traj[i + 1:] = traj[i]
        break
      pos_err = float(np.linalg.norm(d.qpos[:3] - motion.root_pos[i + 1]))
      tilt = quat_angle_deg(d.qpos[3:7], motion.root_quat[i + 1])
      worst_pos, worst_tilt = max(worst_pos, pos_err), max(worst_tilt, tilt)
      t = (i + 1) / FPS
      if fell_at is None and (pos_err > self.FALL_POS_M or tilt > self.FALL_TILT_DEG):
        fell_at = t
      if wobble_at is None and (pos_err > self.WOBBLE_POS_M or tilt > self.WOBBLE_TILT_DEG):
        wobble_at = t
    verdict = "FALLS" if fell_at is not None else "WOBBLES" if wobble_at is not None else "HOLDS UP"
    return dict(
      verdict=verdict, fell_at=fell_at, wobble_at=wobble_at,
      worst_pos_cm=round(100 * worst_pos, 2), worst_tilt_deg=round(worst_tilt, 1),
      traj=traj.astype(np.float32),
    )


PHYS = PhysicsGate()


# ============================================================================ contact / support analysis
CONTACT_Z = 0.006  # a foot (bottom) or hull point this close to the ground is touching it


def _hull_world_points(d) -> np.ndarray:
  pts = []
  for g, p in KIN.hull_points.items():
    b = KIN.model.geom_bodyid[g]
    pts.append(d.xpos[b] + p @ d.xmat[b].reshape(3, 3).T)
  return np.concatenate(pts)


def _support_margin(com_xy, pts_xy) -> float:
  """Signed distance (m) from the COM to the support polygon; > 0 inside."""
  if len(pts_xy) == 0:
    return float("nan")
  if len(pts_xy) >= 3:
    try:
      hull = ConvexHull(pts_xy)
      return float(-np.max(hull.equations[:, :2] @ com_xy + hull.equations[:, 2]))
    except QhullError:
      pass  # collinear: fall through to segment distance
  # a point or a line: always outside, by the distance to it
  if len(pts_xy) == 1:
    return -float(np.linalg.norm(com_xy - pts_xy[0]))
  a, b = pts_xy[np.argmin(pts_xy[:, 0])], pts_xy[np.argmax(pts_xy[:, 0])]
  ab = b - a
  s = np.clip(((com_xy - a) @ ab) / max(ab @ ab, 1e-12), 0, 1)
  return -float(np.linalg.norm(com_xy - (a + s * ab)))


def contact_analysis(motion: md.Motion) -> dict:
  """Per-frame feet, contacts, lowest body point and static support margin."""
  T = len(motion.root_pos)
  feet = np.zeros((T, 4, 3))
  lowest = np.zeros(T)
  margin = np.zeros(T)
  d = KIN.data
  for i in range(T):
    KIN.set_state(motion.root_pos[i], motion.root_quat[i], motion.joint_pos[i])
    feet[i] = [d.site_xpos[KIN.foot_site[leg]] for leg in LEGS]
    hp = _hull_world_points(d)
    lowest[i] = hp[:, 2].min()
    on_feet = feet[i][feet[i][:, 2] - md.FOOT_RADIUS < CONTACT_Z][:, :2]
    on_body = hp[hp[:, 2] < CONTACT_Z][::4, :2]
    margin[i] = _support_margin(d.subtree_com[KIN.base_body][:2], np.concatenate([on_feet, on_body]))
  return dict(feet=feet, lowest_body=lowest, support_margin=margin,
              on=(feet[..., 2] - md.FOOT_RADIUS) < CONTACT_Z)


def slide_stats(feet: np.ndarray, on: np.ndarray, fps: float = FPS) -> dict:
  """Foot sliding: horizontal motion of feet while they touch the ground."""
  v = np.linalg.norm(np.diff(feet[..., :2], axis=0), axis=-1) * fps  # (T-1, 4)
  both = on[1:] & on[:-1]
  worst_span, worst_leg, worst_t = 0.0, None, None
  for k in range(4):
    start = None
    for i in range(len(on) + 1):
      inside = i < len(on) and on[i, k]
      if inside and start is None:
        start = i
      elif not inside and start is not None:
        seg = feet[start:i, k, :2]
        drift = float(np.linalg.norm(seg - seg[0], axis=1).max())
        if drift > worst_span:
          worst_span, worst_leg, worst_t = drift, LEGS[k], (start / fps, (i - 1) / fps)
        start = None
  return dict(
    slide_mean_cm_s=round(100 * float(v[both].mean()) if both.any() else 0.0, 2),
    slide_worst_cm=round(100 * worst_span, 2), slide_worst_leg=worst_leg, slide_worst_t=worst_t,
  )


# ============================================================================ the gate (same for every option)
GATE = dict(
  max_joint_speed=15.0,  # rad/s (AGENTS.md: under about 15)
  max_limit_fraction=0.02,  # joint samples at a limit
  min_lowest=-0.003,  # m: nothing more than 3 mm into the ground
  max_slide_cm=3.0,  # a planted foot may drift at most this much
  max_unbalanced_s=0.3,  # centre of mass outside the support polygon (hops are short)
  allow_wobble=True,  # WOBBLES passes: a trained tracker tightens it up
)


def _longest_run(mask: np.ndarray) -> tuple[int, int]:
  """(length, start) of the longest run of True."""
  best, start, run_start = 0, 0, None
  for i, v in enumerate(list(mask) + [False]):
    if v and run_start is None:
      run_start = i
    elif not v and run_start is not None:
      if i - run_start > best:
        best, start = i - run_start, run_start
      run_start = None
  return best, start


def gate(motion: md.Motion, contacts: dict | None = None) -> dict:
  """Accept/reject a final motion; reasons are written for Gemini to act on."""
  rep = md.check_motion(motion)
  c = contacts or contact_analysis(motion)
  reasons = []
  if rep["lowest_foot_bottom_m"] < GATE["min_lowest"]:
    i = int(np.argmin(c["feet"][..., 2].min(1)))
    reasons.append(f"a foot goes {-100 * rep['lowest_foot_bottom_m']:.1f} cm into the ground around t={i / FPS:.2f} s")
  if rep["lowest_body_point_m"] < GATE["min_lowest"]:
    i = int(np.argmin(c["lowest_body"]))
    reasons.append(f"the body goes {-100 * rep['lowest_body_point_m']:.1f} cm into the ground around t={i / FPS:.2f} s")
  if rep["max_joint_speed_rad_s"] > GATE["max_joint_speed"]:
    jv = np.abs(np.diff(motion.joint_pos, axis=0)) * motion.fps
    i, j = np.unravel_index(np.argmax(jv), jv.shape)
    reasons.append(
      f"joint {JOINT_NAMES[j]} moves at {jv[i, j]:.0f} rad/s around t={i / FPS:.2f} s"
      f" (limit {GATE['max_joint_speed']:.0f}): give that move more time"
    )
  if rep["fraction_of_joint_samples_at_limit"] > GATE["max_limit_fraction"]:
    at = (motion.joint_pos <= KIN.lower + 1e-3) | (motion.joint_pos >= KIN.upper - 1e-3)
    j = int(np.argmax(at.sum(0)))
    i = int(np.argmax(at[:, j]))
    reasons.append(f"joint {JOINT_NAMES[j]} sits at its limit from t={i / FPS:.2f} s: the pose is out of reach")
  s = slide_stats(c["feet"], c["on"])
  if s["slide_worst_cm"] > GATE["max_slide_cm"]:
    a, b = s["slide_worst_t"]
    reasons.append(
      f"foot {s['slide_worst_leg']} slides {s['slide_worst_cm']:.1f} cm on the ground between"
      f" t={a:.2f} and {b:.2f} s: lift it (dz >= 0.02) to move it"
    )
  n_out, i_out = _longest_run(np.nan_to_num(c["support_margin"], nan=-1.0) < -0.01)
  if n_out / FPS > GATE["max_unbalanced_s"]:
    reasons.append(
      f"Pupper is off balance for {n_out / FPS:.2f} s from t={i_out / FPS:.2f} s: its centre of"
      f" mass is up to {-100 * np.nanmin(c['support_margin'][i_out:i_out + n_out]):.1f} cm outside"
      " the feet on the ground. Keep 3 feet down and shift the body ~2 cm away from a foot"
      " before lifting it"
    )
  phys = PHYS.replay(motion)
  bad = {"FALLS"} if GATE["allow_wobble"] else {"FALLS", "WOBBLES"}
  if phys["verdict"] in bad:
    at = phys["fell_at"] or phys["wobble_at"]
    hint = ""
    out = np.where(c["support_margin"] < -0.005)[0]
    if len(out):
      hint = (
        f"; the centre of mass is outside the feet on the ground from t={out[0] / FPS:.2f} s"
        f" (by up to {-100 * np.nanmin(c['support_margin']):.1f} cm): shift the body away from"
        " lifted feet before lifting them, keep 3 feet down"
      )
    reasons.append(f"in a physics replay the robot {phys['verdict'].lower().replace('falls', 'falls over')} at t={at:.2f} s{hint}")
  return dict(ok=not reasons, reasons=reasons, physics=phys, report=rep, slide=s)


# ============================================================================ finishing touches (same for every option)
def stand_deviation_cm(motion: md.Motion, i: int) -> float:
  """How far frame i is from a stand (at that frame's x, y, yaw), in cm-ish units."""
  rpy = quat_to_rpy(motion.root_quat[i])
  ref = md.stand(0.0, x=motion.root_pos[i][0], y=motion.root_pos[i][1], yaw=rpy[2])
  got = KIN.feet(motion.root_pos[i], motion.root_quat[i], motion.joint_pos[i])
  feet = max(np.linalg.norm(got[leg] - np.array(ref.feet[leg])) for leg in LEGS)
  return max(100 * feet, 100 * abs(motion.root_pos[i][2] - H), max(abs(rpy[0]), abs(rpy[1])) / 2)


def _pose_keyframe(motion: md.Motion, i: int, t: float) -> md.Keyframe:
  rp, rq, jq = motion.root_pos[i], motion.root_quat[i], motion.joint_pos[i]
  feet = KIN.feet(rp, rq, jq)
  return md.Keyframe(t, tuple(rp), tuple(quat_to_rpy(rq)), feet={k: tuple(v) for k, v in feet.items()})


def with_stand_ends(motion: md.Motion, blend_s: float = 0.6, tol_cm: float = 1.0) -> tuple[md.Motion, int]:
  """Blend from a stand into the motion and back out, where it doesn't already
  start/end standing. Returns the motion and how many frames were prepended."""
  pre = 0
  if stand_deviation_cm(motion, 0) > tol_cm:
    first = _pose_keyframe(motion, 0, blend_s)
    rpy = quat_to_rpy(motion.root_quat[0])
    tr = md.build_motion([md.stand(0.0, x=first.root_pos[0], y=first.root_pos[1], yaw=rpy[2]), first], fps=FPS)
    motion = concat_motions(tr, motion)
    pre = len(tr.root_pos) - 1
  if stand_deviation_cm(motion, -1) > tol_cm:
    last = _pose_keyframe(motion, len(motion.root_pos) - 1, 0.0)
    rpy = quat_to_rpy(motion.root_quat[-1])
    tr = md.build_motion([last, md.stand(blend_s, x=last.root_pos[0], y=last.root_pos[1], yaw=rpy[2])], fps=FPS)
    motion = concat_motions(motion, tr)
  return motion, pre


def auto_slow(motion: md.Motion, max_speed: float | None = None) -> tuple[md.Motion, float]:
  """Stretch time if a joint is too fast (up to 2x)."""
  max_speed = max_speed or GATE["max_joint_speed"]
  speed = md.check_motion(motion)["max_joint_speed_rad_s"]
  if speed <= max_speed:
    return motion, 1.0
  f = min(2.0, speed / (0.9 * max_speed))
  return stretch_motion(motion, f), f


# ============================================================================ quality metrics (same for every option)
def evaluate(motion: md.Motion, phys: dict | None = None) -> dict:
  """Objective numbers for one final motion. Lower is better unless noted."""
  rep = md.check_motion(motion)
  c = contact_analysis(motion)
  s = slide_stats(c["feet"], c["on"])
  dt = 1.0 / motion.fps
  q = motion.joint_pos
  acc = np.diff(q, 2, axis=0) / dt**2
  jerk = np.diff(q, 3, axis=0) / dt**3
  rpy = rpy_series(motion)
  lift = (c["feet"][..., 2] - md.FOOT_RADIUS).max(0)
  floating = (~c["on"].any(1)) & (c["lowest_body"] > CONTACT_Z)
  phys = phys or PHYS.replay(motion)
  m = c["support_margin"]
  return dict(
    duration_s=rep["duration_s"],
    max_joint_speed=rep["max_joint_speed_rad_s"],
    joint_limit_frac=rep["fraction_of_joint_samples_at_limit"],
    lowest_foot_cm=round(100 * rep["lowest_foot_bottom_m"], 2),
    lowest_body_cm=round(100 * rep["lowest_body_point_m"], 2),
    slide_mean_cm_s=s["slide_mean_cm_s"],
    slide_worst_cm=s["slide_worst_cm"],
    floating_frac=round(float(floating.mean()), 3),
    acc_rms=round(float(np.sqrt((acc**2).mean())), 1),  # rad/s^2
    jerk_rms=round(float(np.sqrt((jerk**2).mean())), 0),  # rad/s^3: smoothness
    com_outside_frac=round(float(np.mean(np.nan_to_num(m, nan=-1) < 0)), 3),
    min_support_margin_cm=round(100 * float(np.nanmin(m)), 2) if np.isfinite(m).any() else None,
    # how much it moves (higher = more motion; catches "does nothing")
    joint_travel_rad=round(float(np.abs(np.diff(q, axis=0)).sum()), 1),
    body_height_range_cm=round(100 * float(np.ptp(motion.root_pos[:, 2])), 1),
    body_tilt_range_deg=round(float(max(np.ptp(rpy[:, 0]), np.ptp(rpy[:, 1]))), 1),
    body_yaw_range_deg=round(float(np.ptp(rpy[:, 2])), 1),
    max_foot_lift_cm=round(100 * float(lift.max()), 1),
    physics=phys["verdict"],
    physics_pos_cm=phys["worst_pos_cm"],
    physics_tilt_deg=phys["worst_tilt_deg"],
  )

In [ ]:
# @title UniMate client and post-processing (options 2 and 3)
import os
import subprocess
import threading
from time import perf_counter

WORKER_PATH = "/content/pupper_bench_worker.py"


class BenchUniMate:
  """Talks to pupper_bench_worker.py (UniMate's environment) over JSON lines.

  Same lifecycle as the course's U.UniMate: the model loads once and stays on
  the GPU, as it would on a generation server, so latency below is per request.
  """

  def __init__(self, unimate_dir="/content/UniMate", worker=WORKER_PATH):
    self.dir = Path(unimate_dir).resolve()
    self.log = self.dir / "bench_worker.log"
    python = self.dir / ".venv" / "bin" / "python"
    if not python.exists():
      raise FileNotFoundError(f"{python} is missing: run the Install UniMate cell first")
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "VIRTUAL_ENV")}
    env["MPLBACKEND"] = "Agg"
    env["PUPPER_UNIMATE_PY"] = str(Path(U.__file__).resolve().parent)
    self.proc = subprocess.Popen(
      [str(python), "-u", worker, "--unimate", str(self.dir)],
      env=env, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
      stderr=self.log.open("w"), text=True, bufsize=1,
    )
    self.lock = threading.Lock()
    self._read()
    self.call(cmd="register", **{k: SK[k] for k in ("name", "names", "clean", "parents", "rest", "face")})
    self.n_out = 0

  def _read(self) -> dict:
    line = self.proc.stdout.readline()
    if not line:
      tail = "\n".join(self.log.read_text().splitlines()[-25:])
      hint = ("\n\nUniMate's environment is missing packages: the install didn't finish."
              " Rerun the 'Install UniMate' cell and read its error.") if "ModuleNotFoundError" in tail else ""
      raise RuntimeError(f"the UniMate worker stopped; the end of {self.log}:\n{tail}{hint}")
    res = json.loads(line)
    if not res.get("ok"):
      raise RuntimeError(f"UniMate: {res.get('error')} (details in {self.log})")
    return res

  def call(self, **req) -> dict:
    if self.proc.poll() is not None:
      raise RuntimeError("UniMate is stopped: run the cell that starts it again")
    with self.lock:
      self.proc.stdin.write(json.dumps(req) + "\n")
      self.proc.stdin.flush()
      return self._read()

  def sample(self, prompts, n=4, seed=0, cfg=3.0, steps=50, solver="euler", frames=60,
             guide=None, guide_yaw=None, pins=(), pin_facing=True, release_t=1.0,
             decode="ric", amp=False, tf32=True, overlap=10):
    """Rig positions (n, T, 19, 3) at 30 fps (robot frame, ground at z = 0), and timing."""
    self.n_out += 1
    out = self.dir / "outputs" / f"bench_{self.n_out:05d}.npz"
    req = dict(
      cmd="sample", skeleton=SK["name"], prompts=list(prompts), n=int(n), seed=seed,
      cfg=float(cfg), steps=int(steps), solver=solver, frames=int(frames),
      pin=[int(p) for p in pins], pin_facing=bool(pin_facing), release_t=float(release_t),
      decode=decode, amp=bool(amp), tf32=bool(tf32), overlap=int(overlap), out=str(out),
    )
    if guide is not None:
      req["dense"] = np.round(np.asarray(guide), 5).tolist()
      req["dense_yaw"] = np.asarray(guide_yaw, float).tolist()
    t0 = perf_counter()
    res = self.call(**req)
    res["wall_s"] = perf_counter() - t0  # includes JSON + file I/O
    return np.load(out)["positions"], res

  def selftest(self, prompt="An object turns around to the left.", seed=0) -> dict:
    return self.call(cmd="selftest", skeleton=SK["name"], prompt=prompt, seed=seed,
                     pelvis=RIG_IX["pelvis"], chest=RIG_IX["chest"])

  def close(self):
    if self.proc.poll() is None:
      self.proc.kill()


# ---------------------------------------------------------------------------- post-processing
def rig_base_xy(raw: np.ndarray) -> np.ndarray:
  """Body centre (mean of the four hip/shoulder joints), as U.fit_pupper defines it."""
  return raw[:, RIG_ROOTS, :2].mean(1)


def correct_root_xy(raw: np.ndarray, guide: np.ndarray, pins) -> np.ndarray:
  """Shift the whole rig so the body is where the keyframes put it at pinned frames.

  The pinned channels fix every joint relative to the body, the height and the
  heading, but not where the body is on the floor (UniMate integrates that from
  velocities). The residual at the pinned frames is interpolated over time and
  removed; foot_lock() cleans up any slide this introduces.
  """
  pins = [p for p in pins if p < len(guide) and p < len(raw)]
  if not pins:
    return raw
  res = rig_base_xy(guide)[pins] - rig_base_xy(raw)[pins]
  fr = np.arange(len(raw))
  off = np.stack([np.interp(fr, pins, res[:, k]) for k in range(2)], -1)
  out = raw.copy()
  out[..., :2] += off[:, None]
  return out


def foot_lock(raw: np.ndarray, z_on=0.012, v_on=0.3, min_frames=3, blend=3,
              guide: np.ndarray | None = None, pins=()) -> np.ndarray:
  """Pin feet that UniMate puts on the ground: no sliding, no sinking.

  A foot is in contact where it is within z_on of the ground and slower than
  v_on (m/s). Each contact of at least min_frames is held still, flat on the
  ground, with a short blend in and out: at the keyframe's foot position if the
  contact contains a keyframe (guide + pins, option 2), else at its median
  position. Only foot joints change, which are what U.fit_pupper's IK follows.
  """
  out = raw.copy()
  T = len(raw)
  pins = sorted(p for p in pins if guide is not None and p < len(guide) and p < T)
  pinset = set(pins)
  for j in RIG_FEET:
    p = raw[:, j]
    v = np.r_[0.0, np.linalg.norm(np.diff(p[:, :2], axis=0), axis=1) * GEN_FPS]
    on = (p[:, 2] < z_on) & (v < v_on)
    for f in pins:  # a keyframe that puts this foot on the ground is a contact
      on[f] |= guide[f, j, 2] < z_on
    i = 0
    while i < T:
      if not on[i]:
        i += 1
        continue
      a = i
      while i < T and on[i]:
        i += 1
      b = i
      if b - a < min_frames:
        continue
      inside = [f for f in pins if a <= f < b and guide[f, j, 2] < z_on]
      anchor = guide[inside[0], j, :2] if inside else np.median(p[a:b, :2], axis=0)
      out[a:b, j, :2] = anchor
      out[a:b, j, 2] = 0.0
      for k in range(1, blend + 1):
        w = 1.0 - k / (blend + 1)
        for f in (a - k, b - 1 + k):
          if 0 <= f < T and not on[f] and f not in pinset:  # pinned frames are constraints
            out[f, j, :2] = w * anchor + (1 - w) * p[f, :2]
            out[f, j, 2] = (1 - w) * p[f, 2]
  return out


SCORE_W = dict(kf=1.0, spike=1.0, slide=0.3, float=0.5, travel=0.5, static=5.0)


def keyframe_spike_cm(raw, guide, key, half_width=6) -> float:
  """How much more sharply the sample changes direction near keyframes than the
  keyframe path does (cm, worst joint): a one-frame dash to meet a pin shows up
  here, a smooth pass through the keyframe doesn't."""
  T = min(len(raw), len(guide))
  frames = sorted({f for k in key for f in range(k - half_width, k + half_width + 1) if 1 <= f < T - 1})
  if not frames:
    return 0.0
  f = np.array(frames)
  d2 = lambda x: np.linalg.norm(x[f - 1] - 2 * x[f] + x[f + 1], axis=-1)  # noqa: E731  (n, 19)
  return 100 * float(np.max(np.maximum(0.0, d2(raw) - d2(guide))))


def sample_score(raw, guide=None, key=(), stride=1.0) -> dict:
  """Cheap ranking of a UniMate sample before the (slow) IK fit. Lower is better.

  keyframe error and spikes at keyframes (cm, option 2) + foot slide (cm/s) +
  floating frames (%) + travel beyond 25 cm (cm) + a penalty for samples that
  barely move. key: the keyframes' frames.
  """
  feet = raw[:, RIG_FEET]
  on = feet[..., 2] < 0.012
  v = np.linalg.norm(np.diff(feet[..., :2], axis=0), axis=-1) * GEN_FPS
  both = on[1:] & on[:-1]
  slide = 100 * float(v[both].mean()) if both.any() else 0.0
  floating = float(np.mean(~on.any(1)))
  base = rig_base_xy(raw)
  travel = stride * float(np.linalg.norm(base - base[0], axis=1).max())
  rel = raw - raw[:, [RIG_IX["spine"]]]
  activity = float(np.linalg.norm(np.diff(rel, axis=0), axis=-1).mean() * GEN_FPS)
  key = [p for p in key if guide is not None and p < len(guide) and p < len(raw)]
  kf = 100 * float(np.linalg.norm(raw[key] - guide[key], axis=-1).mean()) if key else 0.0
  spike = keyframe_spike_cm(raw, guide, key) if key else 0.0
  w = SCORE_W
  score = (w["kf"] * kf + w["spike"] * spike + w["slide"] * slide + w["float"] * 100 * floating
           + w["travel"] * 100 * max(0.0, travel - 0.25) + (w["static"] if activity < 0.01 else 0.0))
  return dict(score=round(score, 3), kf_err_cm=round(kf, 2), spike_cm=round(spike, 2), slide_cm_s=round(slide, 2),
              floating=round(floating, 3), travel_m=round(travel, 3), activity_m_s=round(activity, 4))

In [ ]:
# @title What the LLM is told (read this: it is half of options 1 and 2)
# The robot description uses Pupper's real geometry: the reach numbers below are
# measured with the course IK (md.kinematics) when this cell runs, not guessed.


def _pose_ok(body=(0, 0, 0, 0, 0, 0), feet=None) -> bool:
  """Can IK reach this pose with every target within 2 mm, nothing in the floor,
  no joint at a limit?"""
  kf = KF(0.0, np.array(body, float), feet or {}).md_keyframe()
  root, rq = np.array(kf.root_pos), md.rpy_to_quat(kf.root_rpy)
  q = np.array(DEFAULT_POSE, float)
  for leg in LEGS:
    q, err = KIN.solve_leg(leg, kf.feet[leg], root, rq, q, iters=200)
    if err > 0.002:
      return False
  KIN.set_state(root, rq, q)
  if KIN.lowest_body_point() < -0.001:
    return False
  return not np.any((q <= KIN.lower + 0.02) | (q >= KIN.upper - 0.02))


def _range(vals, test):
  """The contiguous range of vals around 0 that passes test."""
  vals = np.asarray(vals, float)
  ok = np.array([test(v) for v in vals])
  i0 = int(np.argmin(np.abs(vals)))
  lo = hi = i0
  while lo > 0 and ok[lo - 1]:
    lo -= 1
  while hi < len(vals) - 1 and ok[hi + 1]:
    hi += 1
  return float(vals[lo]), float(vals[hi])


def probe_limits() -> dict:
  L = {}
  L["dz"] = _range(np.arange(-0.12, 0.0501, 0.005), lambda v: _pose_ok((0, 0, v, 0, 0, 0)))
  L["x"] = _range(np.arange(-0.08, 0.0801, 0.005), lambda v: _pose_ok((v, 0, -0.02, 0, 0, 0)))
  L["y"] = _range(np.arange(-0.08, 0.0801, 0.005), lambda v: _pose_ok((0, v, -0.02, 0, 0, 0)))
  L["pitch"] = _range(np.arange(-50, 50.1, 2.5), lambda v: _pose_ok((0, 0, -0.03, 0, v, 0)))
  L["roll"] = _range(np.arange(-40, 40.1, 2.5), lambda v: _pose_ok((0, 0, -0.03, v, 0, 0)))
  L["yaw"] = _range(np.arange(-50, 50.1, 2.5), lambda v: _pose_ok((0, 0, -0.03, 0, 0, v)))
  lean = (-0.015, -0.02, -0.01, 0, 0, 0)  # body shifted back-right, as when lifting front_l
  L["lift"] = _range(np.arange(0.0, 0.1201, 0.005),
                     lambda v: _pose_ok(lean, {"front_l": np.array([0, 0, v])}))
  L["lift_dx"] = _range(np.arange(-0.08, 0.0801, 0.005),
                        lambda v: _pose_ok(lean, {"front_l": np.array([v, 0, 0.03])}))
  return L


t0 = time.time()
LIM = probe_limits()
print(f"probed Pupper's reach in {time.time() - t0:.1f} s:")
for k, v in LIM.items():
  print(f"  {k:8} {v[0]:+.3f} .. {v[1]:+.3f}")

_sf = {leg: STAND_FEET[leg][:2].round(3).tolist() for leg in LEGS}
ROBOT_BRIEF = f"""\
## Pupper
A small robot dog: 2-3 kg, about 26 cm long, 12 motors (each leg: abduction, hip, knee). No head,
no tail, no neck: everything expressive comes from the body pose and the legs. Standing, its
body (base_link) is {H:.3f} m above the ground.

## Coordinates (metres, seconds, degrees)
x forward, y left, z up. The robot starts standing at x = y = 0, facing +x.
- body: x, y = body position (m) relative to the start; dz = height change from standing
  (negative = lower); roll, pitch, yaw in degrees.
  pitch > 0 lowers the nose (head down), pitch < 0 raises it.
  roll > 0 raises the left side (leans right); roll < 0 leans left.
  yaw > 0 turns left.
- feet: front_r, front_l, back_r, back_l. Each foot's offset [dx, dy, dz] (m, world frame) from
  where it stands. dz = 0 is on the ground, dz > 0 lifts it. Feet you leave out stay planted.
  Standing foot positions (x, y): {_sf}

## What Pupper can reach (measured with its inverse kinematics; one thing at a time)
- body dz {LIM['dz'][0]:+.3f} to {LIM['dz'][1]:+.3f}; body x {LIM['x'][0]:+.3f} to {LIM['x'][1]:+.3f}; body y {LIM['y'][0]:+.3f} to {LIM['y'][1]:+.3f} (all feet down)
- pitch {LIM['pitch'][0]:+.0f} to {LIM['pitch'][1]:+.0f}, roll {LIM['roll'][0]:+.0f} to {LIM['roll'][1]:+.0f}, yaw {LIM['yaw'][0]:+.0f} to {LIM['yaw'][1]:+.0f} (feet planted, body 3 cm low)
- a lifted front foot: dz up to {LIM['lift'][1]:.3f}, dx {LIM['lift_dx'][0]:+.3f} to {LIM['lift_dx'][1]:+.3f} (at dz 0.03)
Combining several of these at once reaches less; stay well inside the ranges.

## What gets checked before the robot moves (a failed check means the motion is rejected)
- Balance: a 2 kg robot with small round feet. The centre of mass must stay over the feet on
  the ground; it may leave them only briefly (under 0.3 s, e.g. a small hop). Keep 3 feet down.
  Before lifting a foot, shift the body about 2 cm away from it (lifting front_l: body x -0.015,
  y -0.02) and keep that shift while the foot is up. Never lift two feet on the same side.
- Physics: the motion is replayed in a physics simulator; the robot must not fall over.
- Speed: joints stay under 15 rad/s. A big move needs at least 0.3 s; typical moves 0.3-1.0 s.
- No sliding: a foot on the ground must not move sideways. To move a foot, lift it (dz >= 0.02)
  on the way. Nothing may go into the ground.
"""

KEYFRAME_FORMAT = """\
## Keyframe format (JSON)
{"t": 0.0, "pose": "stand"}
{"t": 0.6, "body": {"x": 0, "y": 0, "dz": -0.02, "roll": 0, "pitch": 10, "yaw": 0},
 "feet": {"front_l": [0.0, 0.0, 0.04]}, "ease": "stop", "hold": 0.5}
- t: seconds, starting at 0, increasing, at least 0.1 s apart (a hold counts as time spent).
- body fields default to 0; feet you leave out stay where they stand.
- ease: "stop" (default) = come to rest at this keyframe. "flow" = pass through it without
  stopping (for continuous moves: wiggles, waves, a nod in a series).
- hold: optional seconds to stay in this pose before moving on.
- The first keyframe is {"t": 0, "pose": "stand"} and the last is a stand too.
"""

EXAMPLE_SNEEZE = {
  "plan": "A dramatic sneeze: ah... ah... (nose rises, body leans back), CHOO (nose snaps down, "
          "body drops), a tiny hop on the front feet, settle to a stand.",
  "keyframes": [
    {"t": 0.0, "pose": "stand"},
    {"t": 0.3, "pose": "stand"},
    {"t": 0.75, "body": {"x": -0.01, "dz": 0.004, "pitch": -8}},
    {"t": 1.05, "body": {"x": -0.015, "dz": 0.006, "pitch": -10}},
    {"t": 1.23, "body": {"x": 0.01, "dz": -0.035, "pitch": 22}},
    {"t": 1.43, "body": {"dz": 0.012, "pitch": -6},
     "feet": {"front_r": [0, 0, 0.025], "front_l": [0, 0, 0.025]}},
    {"t": 1.57, "body": {"dz": -0.008, "pitch": 3}},
    {"t": 1.87, "pose": "stand"},
    {"t": 2.37, "pose": "stand"},
  ],
}
EXAMPLE_STEP = {
  "plan": "Step the right hind foot forward and back: lean forward-left first so the other three "
          "feet carry the weight, lift and place the foot, then reverse.",
  "keyframes": [
    {"t": 0.0, "pose": "stand"},
    {"t": 0.5, "pose": "stand"},
    {"t": 0.8, "body": {"x": 0.02, "y": 0.015, "dz": -0.005}},
    {"t": 1.0, "body": {"x": 0.02, "y": 0.015, "dz": -0.005}, "feet": {"back_r": [0.022, 0, 0.04]}},
    {"t": 1.2, "body": {"x": 0.02, "y": 0.015, "dz": -0.005}, "feet": {"back_r": [0.044, 0, 0]}, "hold": 0.5},
    {"t": 1.9, "body": {"x": 0.02, "y": 0.015, "dz": -0.005}, "feet": {"back_r": [0.022, 0, 0.04]}},
    {"t": 2.1, "body": {"x": 0.02, "y": 0.015, "dz": -0.005}},
    {"t": 2.5, "pose": "stand"},
  ],
}

KEYFRAME_SYSTEM = f"""\
You are the motion designer for Pupper, a small robot dog. Someone asked Pupper to do something.
You write the motion as keyframes; they are interpolated, solved with inverse kinematics, and
checked in a physics simulator before the robot moves. If a check fails you get the problems
back and fix them.

{ROBOT_BRIEF}
{KEYFRAME_FORMAT}
## Designing a good motion
- Make it read as the request. Anticipation (a small opposite move first), a clear main action,
  a settle. Living things don't move at constant speed and don't stop dead between every pose:
  use "flow" for poses the body passes through, "stop"/"hold" for poses that should register.
- If the request gives numbers (cm, degrees, seconds, which paw, how many times), match them.
- 1.5 to 8 s in total.

Reply with only the JSON object {{"plan": "...", "keyframes": [...]}}: no prose, no code fences.

## Example: "sneeze"
{json.dumps(EXAMPLE_SNEEZE)}

## Example: "step your right hind foot forward and back"
{json.dumps(EXAMPLE_STEP)}
"""

UNIMATE_PROMPT_RULES = """\
## UniMate prompts
UniMate is a text-to-motion model trained on thousands of animated animals and characters. It
animates Pupper's whole body from one sentence (2 s of motion per sentence). Prompts that work
start with "An object" and describe the motion, not the character: "An object sits down.", not
"A happy puppy sits down on the grass." Examples that work: "An object sits down." / "An object
lies down." / "An object bows down with its chest low and its rear up." / "An object lowers its
body and extends a front paw." / "An object shakes its body." / "An object turns around to the
left." / "An object jumps up and down excitedly."
"""

EXAMPLE_HYBRID_STYLE = {
  "plan": "A mood, carried by two committed key poses: a quick crouch of anticipation, then an eager "
          "perk-up with the nose high. UniMate adds the bouncing and wiggling between and after them.",
  "unimate_prompt": "An object bounces on its front legs and wiggles its body excitedly.",
  "keyframes": [
    {"t": 0.0, "pose": "stand"},
    {"t": 0.5, "body": {"x": -0.01, "dz": -0.03, "pitch": 6}},
    {"t": 1.0, "body": {"x": 0.005, "dz": 0.012, "pitch": -10}},
    {"t": 3.0, "pose": "stand"},
  ],
}
EXAMPLE_HYBRID_PRECISE = {
  "plan": "Exact paw, height and hold time: pin the lean, the lifted paw and the hold; UniMate "
          "fills in how it gets there and back.",
  "unimate_prompt": "An object lifts its front left paw, holds it up, and puts it down.",
  "keyframes": [
    {"t": 0.0, "pose": "stand"},
    {"t": 0.6, "body": {"x": -0.015, "y": -0.02, "dz": -0.01}},
    {"t": 1.0, "body": {"x": -0.015, "y": -0.02, "dz": -0.01}, "feet": {"front_l": [0.01, 0, 0.05]}, "hold": 1.0},
    {"t": 2.4, "body": {"x": -0.015, "y": -0.02, "dz": -0.01}},
    {"t": 3.0, "pose": "stand"},
  ],
}

HYBRID_SYSTEM = f"""\
You direct Pupper, a small robot dog, together with UniMate. Someone asked Pupper to do something.
You write (1) one UniMate prompt and (2) keyframes that UniMate's motion is forced to pass
through. Between keyframes UniMate decides the motion, which looks natural. Your keyframes are
checked with inverse kinematics before UniMate runs, and the result in a physics simulator.

{ROBOT_BRIEF}
{KEYFRAME_FORMAT}
{UNIMATE_PROMPT_RULES}
## How many keyframes: your call, usually a few
- Always the first and last stand; the last keyframe's t is the motion's length (1.5 to 6 s).
- Then the key poses that make the motion read as the request, the ones you would sketch to
  describe it: for a mood or a common animal action usually 2 to 4 (a play bow: the dip and
  the bow itself; excited: a crouch and an eager perk-up). Make them clear and committed: they
  carry the motion. UniMate fills in the transitions, the timing between them and the small
  secondary motion (bounces, wiggles, settling): that is its job, not yours.
- More only where the request needs control: exact amounts (cm, degrees), a particular paw,
  timing or holds, the order of steps, or a pose UniMate might get wrong on a small robot
  (lifting a paw needs the lean first).
- Don't write in-between poses or every swing of a repeated move. Each keyframe pins the whole
  body at that instant, and over-pinning makes it stiff: past about 6 keyframes you are doing
  UniMate's job.
- "ease" doesn't matter here (UniMate decides the motion between keyframes); "hold" does.

Reply with only {{"plan": "...", "unimate_prompt": "An object ...", "keyframes": [...]}}: no prose.

## Example: "act excited!"
{json.dumps(EXAMPLE_HYBRID_STYLE)}

## Example: "lift your front left paw about 5 cm and hold it for a second"
{json.dumps(EXAMPLE_HYBRID_PRECISE)}
"""

TEXT_SYSTEM = f"""\
You turn a request to Pupper, a small robot dog (2 kg, 26 cm long, no head or tail), into prompts
for UniMate, which then animates the robot. The motion is fitted to Pupper and checked in a
physics simulator before the robot moves.

{UNIMATE_PROMPT_RULES}
## Your reply
- "prompts": 1 to 3 prompts. Each is 2 s of motion; they are chained, each starting where the
  previous one ends. Split a sequence ("bow, then wiggle") into one prompt per part.
- "durations": seconds for each prompt (1.5 to 3.0): each prompt's 2 s are stretched to it.
  Slower is gentler and easier to balance.
- Pupper is small and weak: prefer motions in place, keep its feet on the ground, avoid rearing
  up on the hind legs, big jumps, or walking far.

Reply with only {{"plan": "...", "prompts": ["An object ..."], "durations": [2.0]}}: no prose.

## Example: "bow, then hop back up and wiggle"
{{"plan": "Two parts: the bow, then an energetic wiggle.", "prompts": ["An object bows down with its chest low and its rear up.", "An object jumps up and wiggles its body."], "durations": [2.2, 2.0]}}
"""


def parse_json(text: str) -> dict:
  """The JSON object in an LLM reply (code fences and stray prose tolerated)."""
  t = text.strip()
  t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t)
  a, b = t.find("{"), t.rfind("}")
  if a < 0 or b < a:
    raise ValueError("the reply has no JSON object; reply with only the JSON object")
  try:
    obj = json.loads(t[a: b + 1])
  except json.JSONDecodeError as e:
    raise ValueError(f"the reply is not valid JSON ({e}); reply with only the JSON object") from e
  if not isinstance(obj, dict):
    raise ValueError("the reply must be one JSON object")
  return obj


print(f"\nprompt sizes (characters): keyframes {len(KEYFRAME_SYSTEM)}, hybrid {len(HYBRID_SYSTEM)}, text {len(TEXT_SYSTEM)}")

In [ ]:
# @title The test requests, and how precision is measured
# Three kinds, so neither approach is tested only on what it is good at:
#   precise:  numbers, a specific paw, counts, order (keyframes should win)
#   style:    common dog actions and moods (UniMate should win)
#   sequence: several parts in order
# Precise requests have a "spec": a measurement on the final motion, the same
# for every option. Edit freely; ids are used as names in the results.


def _feet_h(motion):
  return feet_series(motion)[..., 2] - md.FOOT_RADIUS  # (T, 4) foot bottom height


def spec_paw_lift(motion, leg, height, hold):
  h = _feet_h(motion)
  k = LEGS.index(leg)
  peak = float(h[:, k].max())
  held = float((h[:, k] > 0.7 * height).sum() / FPS)
  others = float(np.delete(h, k, axis=1).max())
  ok = abs(peak - height) <= 0.4 * height and abs(held - hold) <= 0.4 and others < 0.015
  return dict(ok=ok, measured=f"{leg} up {100 * peak:.1f} cm for {held:.2f} s; other feet up to {100 * others:.1f} cm",
              error=round(abs(peak - height) / height + abs(held - hold) / hold, 3))


def spec_body_drop(motion, depth, hold):
  d = H - motion.root_pos[:, 2]
  peak = float(d.max())
  held = float((d > 0.7 * depth).sum() / FPS)
  lift = float(_feet_h(motion).max())
  ok = abs(peak - depth) <= 0.4 * depth and abs(held - hold) <= 0.4 and lift < 0.015
  return dict(ok=ok, measured=f"down {100 * peak:.1f} cm for {held:.2f} s; feet up to {100 * lift:.1f} cm",
              error=round(abs(peak - depth) / depth + abs(held - hold) / hold, 3))


def spec_lean_lr(motion, deg):
  r = rpy_series(motion)[:, 0]
  left, right = float(-r.min()), float(r.max())  # lean left = roll < 0
  order = int(np.argmin(r)) < int(np.argmax(r))
  ok = abs(left - deg) <= 0.4 * deg and abs(right - deg) <= 0.4 * deg and order
  return dict(ok=ok, measured=f"left {left:.0f} deg, right {right:.0f} deg, {'left first' if order else 'right first'}",
              error=round((abs(left - deg) + abs(right - deg)) / (2 * deg) + (0 if order else 1), 3))


def spec_yaw_lr(motion, deg):
  y = rpy_series(motion)[:, 2]
  left, right = float(y.max()), float(-y.min())  # look left = yaw > 0
  order = int(np.argmax(y)) < int(np.argmin(y))
  ok = abs(left - deg) <= 0.4 * deg and abs(right - deg) <= 0.4 * deg and order
  return dict(ok=ok, measured=f"left {left:.0f} deg, right {right:.0f} deg, {'left first' if order else 'right first'}",
              error=round((abs(left - deg) + abs(right - deg)) / (2 * deg) + (0 if order else 1), 3))


def spec_nod(motion, count, deg):
  p = rpy_series(motion)[:, 1]  # pitch > 0: nose down
  n, armed = 0, True
  for v in p:
    if armed and v > 0.6 * deg:
      n, armed = n + 1, False
    elif v < 0.3 * deg:
      armed = True
  ok = n == count
  return dict(ok=ok, measured=f"{n} nods (nose down past {0.6 * deg:.0f} deg; max {p.max():.0f} deg)",
              error=round(abs(n - count) / count, 3))


SPECS = dict(paw_lift=spec_paw_lift, body_drop=spec_body_drop, lean_lr=spec_lean_lr,
             yaw_lr=spec_yaw_lr, nod=spec_nod)

REQUESTS = [
  # precise
  dict(id="paw", kind="precise", text="Lift your front left paw about 5 cm off the ground, hold it up for one second, then put it back down.",
       spec=("paw_lift", dict(leg="front_l", height=0.05, hold=1.0))),
  dict(id="crouch", kind="precise", text="Crouch down about 4 cm lower than normal, hold it for a second, then stand back up.",
       spec=("body_drop", dict(depth=0.04, hold=1.0))),
  dict(id="lean", kind="precise", text="Lean your body to the left about 15 degrees, then to the right about 15 degrees, then back to the middle.",
       spec=("lean_lr", dict(deg=15))),
  dict(id="nod", kind="precise", text="Nod your body down and up twice, like you're saying yes.",
       spec=("nod", dict(count=2, deg=10))),
  dict(id="look", kind="precise", text="Keep your feet planted and turn your body to look left about 20 degrees, then right about 20 degrees.",
       spec=("yaw_lr", dict(deg=20))),
  # style
  dict(id="bow", kind="style", text="Do a play bow, like you want to play."),
  dict(id="shake", kind="style", text="Shake your body like a wet dog."),
  dict(id="sit", kind="style", text="Sit down like a good dog."),
  dict(id="stretch", kind="style", text="Stretch like you just woke up from a nap."),
  dict(id="excited", kind="style", text="Act excited to see me!"),
  dict(id="sad", kind="style", text="Look sad."),
  # sequence
  dict(id="bow_wiggle", kind="sequence", text="Bow down, then pop back up and wiggle your hips."),
]


def measure_spec(req, motion):
  if not req.get("spec"):
    return None
  name, kw = req["spec"]
  return SPECS[name](motion, **kw)


print(len(REQUESTS), "requests:", ", ".join(r["id"] for r in REQUESTS))

In [ ]:
# @title The three pipelines
# Each takes a request (the words Gemini Live would hand a tool) and returns a
# record: the final 50 Hz motion, the gate verdict, quality metrics, and the
# wall-clock time of every stage. Latency = request in -> motion ready to play
# (excluding rate-limit waits, the robot's controller start (~1.5 s, the same
# for all), and the Live conversation turn before the tool call).
import contextlib
import traceback


@dataclasses.dataclass
class Config:
  # --- option 1: keyframes + interpolation
  kf_interp: str = "spline"  # "spline" (stop/flow keyframes) or "minjerk" (md.build_motion as in the Tricks notebook)
  kf_max_attempts: int = 3  # LLM calls, counting retries after failed checks
  # --- UniMate, shared by options 2 and 3
  n_samples: int = 4  # samples per UniMate call (one batch on the GPU)
  cfg: float = 3.0  # classifier-free guidance (UniMate's default)
  top_k: int = 2  # samples fitted and gated per round, best-ranked first
  rounds: int = 2  # new seeds when no sample passes
  foot_lock: bool = True
  amp: bool = False  # bf16 sampling (faster; check quality before trusting it)
  tf32: bool = True  # TF32 matmuls: ~2x faster on A100/Blackwell (5d); PyTorch's default is off
  # --- option 2: keyframes + UniMate in-betweening
  hybrid_max_attempts: int = 2  # LLM calls if keyframes fail the IK checks
  steps_hybrid: int = 25  # Euler steps (pinning needs a fixed-step solver); see 5d / section 6
  pin_radius: int = 2  # also pin this many frames each side of a keyframe (from the guide):
  #                      a single pinned frame invites a one-frame spike
  pin_holds: bool = True  # pin every frame of a hold (two keyframes with the same pose)
  release_t: float = 1.0  # stop pinning keyframes after this flow time (1.0 = never)
  pin_facing: bool = True
  decode_hybrid: str = "ric"  # "ric": positions (honours the pins); "fk": rotations
  # --- option 3: UniMate text-to-motion
  solver_text: str = "euler"  # or "dopri5", UniMate's default adaptive solver (~6x slower, 5d)
  steps_text: int = 25  # Euler steps; the same as option 2 by default
  decode_text: str = "fk"  # as the course notebook
  stride: float = 0.6  # U.fit_pupper's step/travel scale (course default)
  seed: int = 0


CFG = Config()


class Stopwatch:
  def __init__(self):
    self.t0 = perf_counter()
    self.stages: dict[str, float] = {}
    self.waited = 0.0
    self.llm_calls: list[float] = []

  @contextlib.contextmanager
  def __call__(self, stage: str):
    s = perf_counter()
    try:
      yield
    finally:
      self.stages[stage] = self.stages.get(stage, 0.0) + perf_counter() - s

  def llm(self, system: str, user: str) -> str:
    s = perf_counter()
    text, seconds, waited = llm(system, user)
    self.waited += waited
    self.llm_calls.append(seconds)
    self.stages["llm"] = self.stages.get("llm", 0.0) + (perf_counter() - s - waited)
    return text

  def total(self) -> float:
    return perf_counter() - self.t0 - self.waited


def feedback(user: str, text: str, errs: list[str]) -> str:
  return (
    f"{user}\n\nYour previous answer:\n{text.strip()}\n\nIt failed these checks:\n"
    + "\n".join(f"- {e}" for e in errs)
    + "\n\nFix every problem and reply with the complete corrected JSON object only."
  )


def reach_problems(motion: md.Motion, kfs: list[KF]) -> list[str]:
  """Keyframe targets the IK couldn't reach (> 2 mm)."""
  out = []
  for e in keyframe_errors(motion, kfs):
    leg, err = max(e["foot_err"].items(), key=lambda kv: kv[1])
    if err > 0.002:
      out.append(f"keyframe {e['src']} (t={e['t']:.2f}): foot {leg} can't reach its target"
                 f" (off by {100 * err:.1f} cm): move it closer or change the body pose")
  return out


def finalize(motion: md.Motion):
  """Same for every option: blend in/out of a stand if needed, slow down if too fast."""
  motion, pre = with_stand_ends(motion)
  motion, slow = auto_slow(motion)
  return motion, pre, slow


def _record(option, req, sw, motion, g, **extra) -> dict:
  rec = dict(
    option=option, request_id=req["id"], request=req["text"], kind=req["kind"],
    ok=bool(g["ok"]) if g else False, reasons=g["reasons"] if g else ["no motion"],
    latency=dict(sw.stages), total_s=sw.total(), llm_calls=list(sw.llm_calls),
    motion=motion_to_dict(motion) if motion is not None else None,
    replay=g["physics"]["traj"] if g else None,
    metrics=evaluate(motion, g["physics"]) if motion is not None else None,
    spec=measure_spec(req, motion) if motion is not None else None,
  )
  rec.update(extra)
  return rec


def _kf_adherence(motion, kfs):
  if motion is None or not kfs:
    return None
  e = keyframe_errors(motion, kfs)
  return dict(
    foot_cm_mean=round(100 * float(np.mean([v for x in e for v in x["foot_err"].values()])), 2),
    foot_cm_max=round(100 * float(max(v for x in e for v in x["foot_err"].values())), 2),
    root_cm_max=round(100 * float(max(x["root_err"] for x in e)), 2),
    rot_deg_max=round(float(max(x["rot_err"] for x in e)), 1),
  )


def _kfs_json(kfs):
  return [dict(t=k.t, body=k.body.tolist(), feet={l: v.tolist() for l, v in k.feet.items()}, ease=k.ease) for k in kfs]


# ---------------------------------------------------------------------------- option 1
def run_keyframes(req, cfg: Config = CFG) -> dict:
  sw = Stopwatch()
  user = f'Request: "{req["text"]}"'
  attempts, best = [], None
  for _ in range(cfg.kf_max_attempts):
    text = sw.llm(KEYFRAME_SYSTEM, user)
    errs, kfs, motion, g = [], [], None, None
    try:
      data = parse_json(text)
      kfs, errs = parse_keyframes(data.get("keyframes"))
    except ValueError as e:
      errs = [str(e)]
    if not errs:
      with sw("build"):
        motion = build_from_keyframes(kfs, cfg.kf_interp)
      with sw("verify"):
        errs = reach_problems(motion, kfs)
        g = gate(motion)
        errs += g["reasons"]
    attempts.append(dict(text=text, errors=errs))
    if motion is not None and (best is None or len(errs) < len(best[3])):
      best = (motion, kfs, g, errs)
    if not errs:
      break
    user = feedback(f'Request: "{req["text"]}"', text, errs)
  if best is None:
    return _record("1_keyframes", req, sw, None, None, attempts=attempts)
  motion, kfs, g, errs = best
  built = motion
  with sw("finalize"):
    motion, pre, slow = finalize(motion)
  if pre or slow != 1.0:
    with sw("gate"):
      g = gate(motion)
  if errs and g["ok"]:  # IK-reach problems aren't in the gate; keep them as a failure
    g = dict(g, ok=False, reasons=errs)
  return _record("1_keyframes", req, sw, motion, g, attempts=attempts, keyframes=_kfs_json(kfs),
                 kf_adherence=_kf_adherence(built, kfs), pre_frames=pre, slow=slow)


# ---------------------------------------------------------------------------- option 2
def pin_frames(kfs: list[KF], n_frames: int, radius: int = 0, holds: bool = True) -> tuple[list[int], list[int]]:
  """Keyframe frames at 30 fps, and what gets pinned: those frames +- radius, plus
  (holds=True) every frame of a hold. UniMate only has to match pinned frames, so
  pinning just a hold's two ends lets it leave the pose in between and come back
  for one frame at each end."""
  clamp = lambda f: min(n_frames - 1, max(0, f))  # noqa: E731
  key = sorted({clamp(int(round(k.t * GEN_FPS))) for k in kfs})
  pins = {clamp(f + d) for f in key for d in range(-radius, radius + 1)}
  if holds:
    # Only explicit "hold"s (parse_keyframes expands one into two keyframes with the
    # same source index). Two equal poses in a row are NOT a hold: "stand ... stand"
    # around a style request would otherwise pin the whole motion to standing.
    for a, b in zip(kfs, kfs[1:]):
      if a.src >= 0 and a.src == b.src:
        pins.update(range(clamp(int(round(a.t * GEN_FPS))), clamp(int(round(b.t * GEN_FPS))) + 1))
  return key, sorted(pins)


def plan_hybrid(req, sw: Stopwatch, cfg: Config = CFG) -> tuple[dict, list, list]:
  """The LLM half of option 2: a UniMate prompt + keyframes that pass the IK checks."""
  user = f'Request: "{req["text"]}"'
  attempts, notes = [], []
  for _ in range(cfg.hybrid_max_attempts):
    text = sw.llm(HYBRID_SYSTEM, user)
    errs, kfs, prompt = [], [], ""
    try:
      data = parse_json(text)
      kfs, errs = parse_keyframes(data.get("keyframes"))
      prompt = str(data.get("unimate_prompt") or "").strip()
      if not prompt:
        errs.append("unimate_prompt is missing: one sentence starting with \"An object\"")
    except ValueError as e:
      errs = [str(e)]
    if not errs:
      with sw("build"):
        guide_motion = build_from_keyframes(kfs, "spline")
      with sw("verify"):
        errs = reach_problems(guide_motion, kfs)
        if guide_motion.report["fraction_of_joint_samples_at_limit"] > GATE["max_limit_fraction"]:
          errs.append("a joint sits at its limit in your keyframes: a pose is out of reach")
    attempts.append(dict(text=text, errors=errs))
    if not errs:
      return dict(prompt=prompt, kfs=kfs, guide_motion=guide_motion, text=text), attempts, notes
    user = feedback(f'Request: "{req["text"]}"', text, errs)
  # Keyframes never passed: fall back to pinning start and end only, as a tool would.
  notes.append("keyframes rejected after retries: pinned start and end only")
  try:
    prompt = str(parse_json(attempts[-1]["text"]).get("unimate_prompt") or "").strip()
  except ValueError:
    prompt = ""
  prompt = prompt or "An object moves happily."
  kfs = parse_keyframes([{"t": 0, "pose": "stand"}, {"t": 3.0, "pose": "stand"}])[0]
  with sw("build"):
    guide_motion = build_from_keyframes(kfs, "spline")
  return dict(prompt=prompt, kfs=kfs, guide_motion=guide_motion, text=None), attempts, notes


def run_hybrid(req, cfg: Config = CFG, plan: dict | None = None, um=None) -> dict:
  """plan: reuse an earlier plan_hybrid() result (tuning UniMate settings without new LLM calls)."""
  um = um or UM
  sw = Stopwatch()
  attempts, notes = [], []
  if plan is None:
    plan, attempts, notes = plan_hybrid(req, sw, cfg)
  prompt, kfs, guide_motion = plan["prompt"], plan["kfs"], plan["guide_motion"]

  with sw("guide"):
    guide, gyaw = unimate_guide(guide_motion)
    N = len(guide)
    key, pins = pin_frames(kfs, N, cfg.pin_radius, cfg.pin_holds)
  best, gen_info, ranked = None, [], []
  for rnd in range(cfg.rounds):
    with sw("unimate"):
      raw, info = um.sample([prompt], n=cfg.n_samples, seed=cfg.seed + 1000 * rnd, cfg=cfg.cfg,
                            steps=cfg.steps_hybrid, solver="euler", frames=N, guide=guide,
                            guide_yaw=gyaw, pins=pins, pin_facing=cfg.pin_facing,
                            release_t=cfg.release_t, decode=cfg.decode_hybrid, amp=cfg.amp, tf32=cfg.tf32)
    gen_info.append(info)
    with sw("rank"):
      cands = [correct_root_xy(r, guide, key) for r in raw]
      scores = [sample_score(c, guide, key, 1.0) for c in cands]
      order = list(np.argsort([s["score"] for s in scores]))
    ranked.append(scores)
    for idx in order[: cfg.top_k]:
      with sw("fit"):
        r = foot_lock(cands[idx], guide=guide, pins=pins) if cfg.foot_lock else cands[idx]
        fit = U.fit_pupper(r, stride=1.0, trim_end=(len(r) - N) / GEN_FPS)
      with sw("finalize"):
        motion, pre, slow = finalize(fit.motion)
      with sw("gate"):
        g = gate(motion)
      cand = (motion, g, fit, pre, slow, rnd, int(idx))
      if best is None or (g["ok"] and not best[1]["ok"]) or (g["ok"] == best[1]["ok"] and len(g["reasons"]) < len(best[1]["reasons"])):
        best = cand
      if g["ok"]:
        break
    if best is not None and best[1]["ok"]:
      break
  motion, g, fit, pre, slow, rnd, idx = best
  return _record(
    "2_hybrid", req, sw, motion, g, attempts=attempts, notes=notes, keyframes=_kfs_json(kfs),
    unimate_prompts=[prompt], n_keyframes=len(kfs), pinned_frames=pins,
    kf_adherence=_kf_adherence(fit.motion, kfs), fit_stats=fit.stats, gen=gen_info,
    sample_scores=ranked, chosen=(rnd, idx), skeleton=fit.skeleton.astype(np.float32),
    pre_frames=pre, slow=slow,
  )


# ---------------------------------------------------------------------------- option 3
def plan_text(req, sw) -> tuple[dict, list]:
  user = f'Request: "{req["text"]}"'
  attempts = []
  for _ in range(2):
    text = sw.llm(TEXT_SYSTEM, user)
    errs = []
    try:
      data = parse_json(text)
      prompts = [str(p).strip() for p in data.get("prompts", []) if str(p).strip()]
      if not 1 <= len(prompts) <= 3:
        errs.append("give 1 to 3 prompts")
    except ValueError as e:
      errs = [str(e)]
    attempts.append(dict(text=text, errors=errs))
    if not errs:
      prompts = [p if p.lower().startswith("an object") else "An object " + p[0].lower() + p[1:] for p in prompts]
      dur = data.get("durations") or []
      try:
        dur = [float(d) for d in dur]
      except (TypeError, ValueError):
        dur = []
      if len(dur) != len(prompts):
        dur = [2.0] * len(prompts)
      dur = [float(np.clip(d, 1.5, 3.0)) for d in dur]
      return dict(prompts=prompts, durations=dur, text=text), attempts
    user = feedback(f'Request: "{req["text"]}"', text, errs)
  return dict(prompts=[f"An object {req['text'].rstrip('.!').lower()}."], durations=[2.0], text=None), attempts


def run_text(req, cfg: Config = CFG, plan: dict | None = None, um=None) -> dict:
  um = um or UM
  sw = Stopwatch()
  attempts = []
  if plan is None:
    plan, attempts = plan_text(req, sw)
  prompts, durations = plan["prompts"], plan["durations"]
  frames = U.SEGMENT + (U.SEGMENT - 10) * (len(prompts) - 1)
  best, gen_info, ranked = None, [], []
  for rnd in range(cfg.rounds):
    with sw("unimate"):
      raw, info = um.sample(prompts, n=cfg.n_samples, seed=cfg.seed + 1000 * rnd, cfg=cfg.cfg,
                            steps=cfg.steps_text, solver=cfg.solver_text, frames=frames,
                            decode=cfg.decode_text, amp=cfg.amp, tf32=cfg.tf32)
    gen_info.append(info)
    with sw("rank"):
      scores = [sample_score(r, stride=cfg.stride) for r in raw]
      order = list(np.argsort([s["score"] for s in scores]))
    ranked.append(scores)
    for idx in order[: cfg.top_k]:
      with sw("fit"):
        r = foot_lock(raw[idx]) if cfg.foot_lock else raw[idx]
        fit = U.fit_pupper(r, durations=durations, stride=cfg.stride)
      with sw("finalize"):
        motion, pre, slow = finalize(fit.motion)
      with sw("gate"):
        g = gate(motion)
      cand = (motion, g, fit, pre, slow, rnd, int(idx))
      if best is None or (g["ok"] and not best[1]["ok"]) or (g["ok"] == best[1]["ok"] and len(g["reasons"]) < len(best[1]["reasons"])):
        best = cand
      if g["ok"]:
        break
    if best is not None and best[1]["ok"]:
      break
  motion, g, fit, pre, slow, rnd, idx = best
  return _record(
    "3_unimate", req, sw, motion, g, attempts=attempts, unimate_prompts=prompts,
    durations=durations, fit_stats=fit.stats, gen=gen_info, sample_scores=ranked,
    chosen=(rnd, idx), skeleton=fit.skeleton.astype(np.float32), pre_frames=pre, slow=slow,
  )


PIPELINES = {"1_keyframes": run_keyframes, "2_hybrid": run_hybrid, "3_unimate": run_text}


def run_one(option, req, cfg: Config = CFG, **kw) -> dict:
  try:
    return PIPELINES[option](req, cfg, **kw)
  except Exception as e:  # noqa: BLE001  (one failure shouldn't stop a long run)
    traceback.print_exc()
    return dict(option=option, request_id=req["id"], request=req["text"], kind=req["kind"],
                ok=False, reasons=[f"crashed: {type(e).__name__}: {e}"], error=traceback.format_exc(),
                latency={}, total_s=float("nan"), llm_calls=[], motion=None, replay=None,
                metrics=None, spec=None)

In [ ]:
# @title Start UniMate and warm it up (about 1 minute)
# Loads the model once, as a generation server would. The first calls compile
# CUDA kernels; they are run here so they don't count in the benchmark.
if "UM" in globals():
  UM.close()
t0 = time.time()
UM = BenchUniMate(UNIMATE_DIR)
print(f"UniMate loaded in {time.time() - t0:.0f} s")
for solver in ("dopri5", "euler"):
  for _ in range(2):
    _raw, info = UM.sample(["An object sits down."], n=CFG.n_samples, solver=solver, steps=50)
  print(f"warm {solver:6}: {CFG.n_samples} samples of 2 s in {info['seconds']:.2f} s on the GPU"
        f" ({info['wall_s']:.2f} s including I/O)")

## 5. Self-tests
Run these before believing any number below.

- **5a:** the gate passes motions we know work (a stand, the course's sit and sneeze) and rejects ones that can't
  (two paws up for a second, a move too fast for the motors).
- **5b:** the values we pin really are UniMate's representation (axes, scale, joint order, ground, heading).
- **5c:** UniMate actually passes through pinned keyframes (compared with the same prompt unpinned), and still
  does after the fit to Pupper.

**Optional, about 5 minutes:** compare the fast CPU gate with `play_motion` itself, so you know how far to trust the
gate's verdicts.

In [ ]:
# @title Self-tests: does each piece do what it claims? (about 1 minute)
from mjlab.tasks.tracking.config.pupper.tricks import sit as course_sit


def _check(name, ok, detail=""):
  print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f": {detail}" if detail else ""))
  return ok


print("--- 5a. The gate on motions we know")
stand = md.build_motion([md.stand(0.0), md.stand(2.0)], fps=FPS)
g = gate(stand)
_check("a still stand passes", g["ok"] and g["physics"]["verdict"] == "HOLDS UP", g["physics"]["verdict"])
g = gate(md.build_motion(course_sit.keyframes(), fps=FPS))
_check("the course's sit (a trained trick) passes", g["ok"], f"{g['physics']['verdict']} {g['reasons']}")
k, _ = parse_keyframes(EXAMPLE_SNEEZE["keyframes"])
g = gate(build_from_keyframes(k))
_check("the course's sneeze (a trained trick) passes", g["ok"], f"{g['physics']['verdict']} {g['reasons']}")
k, _ = parse_keyframes([{"t": 0, "pose": "stand"},
                        {"t": 0.5, "feet": {"front_r": [0, 0, 0.06], "front_l": [0, 0, 0.06]}, "hold": 1.0},
                        {"t": 2.0, "pose": "stand"}])
g = gate(build_from_keyframes(k))
_check("both front paws up for 1 s is rejected", not g["ok"], g["reasons"][0][:90] if g["reasons"] else "")
k, _ = parse_keyframes([{"t": 0, "pose": "stand"}, {"t": 0.15, "body": {"dz": -0.07, "pitch": 30}}, {"t": 0.3, "pose": "stand"}])
g = gate(build_from_keyframes(k))
_check("a too-fast move is rejected", not g["ok"], g["reasons"][0][:90] if g["reasons"] else "")

print("\n--- 5b. UniMate encoder: do our pinned values speak UniMate's representation?")
st = UM.selftest()
# Wrong axes, scale or joint order would show up as centimetres, not a millimetre.
_check("positions round-trip (axes, scale, joint order)", st["rifke_roundtrip_err_m"] < 1e-3,
       f"{1000 * st['rifke_roundtrip_err_m']:.3f} mm")
# The model's 6D outputs aren't exactly normalized; a wrong convention would be off by ~1.
_check("6D rotation convention matches", st["sixd_roundtrip_err"] < 0.05, f"{st['sixd_roundtrip_err']:.2e}")
_check("facing convention matches (body heading vs model's)", st["facing_body_vs_model_deg_median"] < 15,
       f"median {st['facing_body_vs_model_deg_median']:.1f} deg, max {st['facing_body_vs_model_deg_max']:.1f} deg,"
       f" the sample turned {st['yaw_turned_deg']:.0f} deg")
_check("ground convention (lowest joint near z = 0)", abs(st["lowest_joint_z_m"]) < 0.02, f"{100 * st['lowest_joint_z_m']:.1f} cm")

print("\n--- 5c. In-betweening: does UniMate pass through the keyframes, and hold them?")
kfs, _ = parse_keyframes(EXAMPLE_HYBRID_PRECISE["keyframes"])  # paw up 5 cm, held 1 s
guide, gyaw = unimate_guide(build_from_keyframes(kfs))
P = [EXAMPLE_HYBRID_PRECISE["unimate_prompt"]]
key, ends_only = pin_frames(kfs, len(guide), radius=0, holds=False)
_, pins = pin_frames(kfs, len(guide), CFG.pin_radius, CFG.pin_holds)  # what the pipeline pins
raw, info = UM.sample(P, n=4, frames=len(guide))
err = [sample_score(correct_root_xy(r, guide, key), guide, key)["kf_err_cm"] for r in raw]
print(f"      unpinned (text only), keyframe error: {np.round(err, 2).tolist()} cm")
for decode in ("fk", "ric"):
  raw, info = UM.sample(P, n=4, frames=len(guide), guide=guide, guide_yaw=gyaw, pins=pins, decode=decode)
  err = [sample_score(correct_root_xy(r, guide, key), guide, key)["kf_err_cm"] for r in raw]
  _check(f"pinned, decode={decode}: keyframe error under 1.5 cm", min(err) < 1.5,
         f"per sample {np.round(err, 2).tolist()} cm")


def _fit_and_measure(r0, pinned):
  """As option 2 does it: shift, foot-lock, fit; then measure the paw hold."""
  r0 = correct_root_xy(r0, guide, key)
  fit = U.fit_pupper(foot_lock(r0, guide=guide, pins=pinned), stride=1.0,
                     trim_end=(len(r0) - len(guide)) / GEN_FPS)
  return sample_score(r0, guide, key), _kf_adherence(fit.motion, kfs), spec_paw_lift(fit.motion, "front_l", 0.05, 1.0)


rows_5c = {}
for name, pinned in (("only the keyframe frames (before)", ends_only),
                     (f"keyframes +-{CFG.pin_radius} frames and whole holds (pipeline)", pins)):
  raw, info = UM.sample(P, n=4, frames=len(guide), guide=guide, guide_yaw=gyaw, pins=pinned)
  print(f"      pinning {name}: {len(pinned)} of {len(guide)} frames")
  rows_5c[name] = []
  for i, r0 in enumerate(raw):
    sc, ka, sp = _fit_and_measure(r0, pinned)
    rows_5c[name].append((sc, ka, sp))
    print(f"        sample {i}: spike {sc['spike_cm']:4.1f} cm | after the fit, feet off by up to"
          f" {ka['foot_cm_max']:4.1f} cm | {sp['measured']}")
best = min(rows_5c[name], key=lambda x: x[0]["score"])  # the sample option 2's ranking would pick
_check("pipeline pinning: the top-ranked sample holds the paw up as asked", best[2]["ok"], best[2]["measured"])
_check("pipeline pinning: its feet are within 2 cm of the keyframes after the fit",
       best[1]["foot_cm_max"] < 2.0, str(best[1]))

print("\n--- 5d. UniMate speed: GPU seconds for 4 samples (quality of each setting: section 6)")
_, a = UM.sample(P, n=4, frames=len(guide), guide=guide, guide_yaw=gyaw, pins=pins, tf32=False)
print(f"      in-betweening, {a['segments']} windows, 50 Euler steps, full fp32: {a['seconds']:6.2f} s")
for steps in (50, 25, 15, 10):
  _, a = UM.sample(P, n=4, frames=len(guide), guide=guide, guide_yaw=gyaw, pins=pins, steps=steps)
  _, b = UM.sample(P, n=4, frames=60, solver="euler", steps=steps)
  print(f"      tf32, {steps:2} Euler steps: in-betweening {a['segments']} windows {a['seconds']:6.2f} s"
        f" | text-only 1 window {b['seconds']:6.2f} s")
_, b = UM.sample(P, n=4, frames=60, solver="dopri5")
print(f"      tf32, dopri5 (UniMate's default solver): text-only 1 window {b['seconds']:6.2f} s")

In [ ]:
# @title (Optional, ~5 min) Calibrate the fast gate against play_motion
# The benchmark's gate replays physics on the CPU in ~0.1 s so it can sit inside
# the request loop. play_motion (the course's GPU replay through mjlab) is the
# reference. If they disagree a lot, trust play_motion and tell me.
import re as _re


def play_motion_verdict(motion: md.Motion, name: str) -> dict:
  d = Path("/content/calibration")  # scratch: these are only for this comparison
  d.mkdir(exist_ok=True)
  path = motion.save_npz(d / f"{name}.npz")
  out = subprocess.run([sys.executable, "-m", "mjlab.tasks.tracking.config.pupper.play_motion", str(path)],
                       capture_output=True, text=True, cwd="/content/pupper-mjlab").stdout
  v = _re.search(r"verdict\s+(HOLDS UP|WOBBLES|FALLS)", out)
  p = _re.search(r"worst body position error\s+([\d.]+) cm", out)
  t = _re.search(r"worst body orientation error\s+([\d.]+) deg", out)
  return dict(verdict=v.group(1) if v else "?", pos_cm=float(p.group(1)) if p else None,
              tilt_deg=float(t.group(1)) if t else None)


calib = {
  "stand": md.build_motion([md.stand(0.0), md.stand(2.0)], fps=FPS),
  "sit": md.build_motion(course_sit.keyframes(), fps=FPS),
  "sneeze": build_from_keyframes(parse_keyframes(EXAMPLE_SNEEZE["keyframes"])[0]),
  "step": build_from_keyframes(parse_keyframes(EXAMPLE_STEP["keyframes"])[0]),
  "two_paws_up": build_from_keyframes(parse_keyframes(
    [{"t": 0, "pose": "stand"}, {"t": 0.5, "feet": {"front_r": [0, 0, 0.06], "front_l": [0, 0, 0.06]}, "hold": 1.0},
     {"t": 2.0, "pose": "stand"}])[0]),
}
_raw, _ = UM.sample(["An object shakes its body."], n=1)
calib["unimate_shake"] = finalize(U.fit_pupper(foot_lock(_raw[0])).motion)[0]
print(f"{'motion':16} {'fast gate':>26} {'play_motion':>26}")
for name, m in calib.items():
  a = PHYS.replay(m)
  b = play_motion_verdict(m, name)
  print(f"{name:16} {a['verdict']:>9} {a['worst_pos_cm']:5.1f} cm {a['worst_tilt_deg']:5.1f} deg"
        f" {b['verdict']:>9} {b['pos_cm'] or float('nan'):5.1f} cm {b['tilt_deg'] or float('nan'):5.1f} deg")

## 6. Tune UniMate first (recommended)
So option 2 or 3 doesn't lose to a bad setting. Each request is planned by the LLM **once**, then every variant runs on
that same plan. Look for high `passed`, low `slide_cm` and, for option 2, low `kf_foot_cm_max`; check `unimate_s` for
the latency cost. Then set the winners, e.g. `CFG = dataclasses.replace(CFG, cfg=5.0, pin_radius=1)`, and rerun the
"Start UniMate" cell's warm-up if you changed the solver.

Option 1's knob is `kf_interp`: "spline" (default; it can pass through "flow" keyframes without stopping) or "minjerk"
(stop at every keyframe, as in the Tricks notebook). To compare them, run section 7 once with each, using different
`RESULTS_DIR`s.

In [ ]:
# @title Tune UniMate's settings on a few requests (~5-10 min)
# Same LLM plan for every variant (planned once per request), so only the
# UniMate settings change. Pick the variant that passes most often with low
# slide / keyframe error, write it into CFG, then run the benchmark.
import pandas as pd

TUNE_REQUESTS = ["paw", "crouch", "bow", "shake", "bow_wiggle"]  # ids from REQUESTS (paw, crouch: holds)
GRID_HYBRID = {
  "default (+-2, holds, 25 steps)": {},
  "cfg 2": dict(cfg=2.0),
  "cfg 5": dict(cfg=5.0),
  "pin keyframe frames only": dict(pin_radius=0, pin_holds=False),
  "pin +-0, holds": dict(pin_radius=0),
  "pin +-4, holds": dict(pin_radius=4),
  "release at 0.9": dict(release_t=0.9),
  "decode fk": dict(decode_hybrid="fk"),
  "50 steps": dict(steps_hybrid=50),
  "15 steps": dict(steps_hybrid=15),
  "10 steps": dict(steps_hybrid=10),
  "bf16": dict(amp=True),
  "bf16, 15 steps": dict(amp=True, steps_hybrid=15),
}
GRID_TEXT = {
  "default (euler 25)": {},
  "dopri5 (UniMate default)": dict(solver_text="dopri5"),
  "cfg 5": dict(cfg=5.0),
  "euler 50 steps": dict(solver_text="euler", steps_text=50),
  "euler 15 steps": dict(solver_text="euler", steps_text=15),
  "euler 10 steps": dict(solver_text="euler", steps_text=10),
  "stride 0.4": dict(stride=0.4),
  "8 samples": dict(n_samples=8),
  "decode ric": dict(decode_text="ric"),
  "bf16, euler 15": dict(amp=True, solver_text="euler", steps_text=15),
}


def _tune_rows(option, grid, plans):
  rows = []
  for name, kw in grid.items():
    cfg = dataclasses.replace(CFG, **kw)
    for rid, plan in plans.items():
      req = next(r for r in REQUESTS if r["id"] == rid)
      rec = run_one(option, req, cfg, plan=plan)
      m = rec["metrics"] or {}
      ka = rec.get("kf_adherence") or {}
      rows.append(dict(
        variant=name, request=rid, passed=rec["ok"], seconds=rec["total_s"],
        unimate_s=rec["latency"].get("unimate", np.nan), slide_cm=m.get("slide_worst_cm"),
        jerk=m.get("jerk_rms"), physics=m.get("physics"), kf_foot_cm_max=ka.get("foot_cm_max"),
        spec_ok=float(rec["spec"]["ok"]) if rec.get("spec") else np.nan,
      ))
      print(f"  {option} | {name:16} | {rid:10} | {'PASS' if rec['ok'] else 'fail'} {rec['total_s']:5.1f} s")
  return pd.DataFrame(rows)


_sw = Stopwatch()
hybrid_plans = {rid: plan_hybrid(next(r for r in REQUESTS if r["id"] == rid), _sw)[0] for rid in TUNE_REQUESTS}
text_plans = {rid: plan_text(next(r for r in REQUESTS if r["id"] == rid), _sw)[0] for rid in TUNE_REQUESTS}
for rid in TUNE_REQUESTS:
  print(f"{rid}: hybrid '{hybrid_plans[rid]['prompt']}' + {len(hybrid_plans[rid]['kfs'])} keyframes | text {text_plans[rid]['prompts']}")

tune_h = _tune_rows("2_hybrid", GRID_HYBRID, hybrid_plans)
tune_t = _tune_rows("3_unimate", GRID_TEXT, text_plans)
agg = dict(passed="mean", spec_ok="mean", seconds="median", unimate_s="median", slide_cm="median",
           jerk="median", kf_foot_cm_max="median")
print("\nOption 2 (hybrid):")
display(tune_h.groupby("variant").agg({k: v for k, v in agg.items() if k in tune_h}).round(2))
print("Option 3 (UniMate text):")
display(tune_t.groupby("variant").agg({k: v for k, v in agg.items() if k != "kf_foot_cm_max"}).round(2))
(RESULTS_ROOT / "tuning").mkdir(exist_ok=True)
_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
tune_h.to_csv(RESULTS_ROOT / "tuning" / f"{_stamp}_hybrid.csv", index=False)
tune_t.to_csv(RESULTS_ROOT / "tuning" / f"{_stamp}_text.csv", index=False)
print("saved to", RESULTS_ROOT / "tuning")

## 7. The benchmark
**Experiments.** Each run of the benchmark is an *experiment*: a folder `runs/<name>/` on Drive with its results, its
settings and the exact prompts the LLM got. A new name starts a new one; picking an existing name continues it, skipping
what is already done. Nothing is overwritten or deleted, so there is no need to touch Drive between runs. To rerun one
option after a change (say, option 2 with new prompts), start a new experiment with only that option; section 8 can
then show it next to the other options' older results.

Options run in a shuffled order for each request, so none of them always goes first (warm caches, rate limits).
`PASS` means the motion made it through the gate. A `fail` still keeps the best attempt, so you can look at it in the viewer.

In [ ]:
# @title Choose what to run
# A new experiment name starts a new experiment folder; picking an existing one
# continues it (runs already done are skipped). Nothing is ever overwritten.
import datetime
import re

import ipywidgets as widgets
from IPython.display import display as _display

_ids = [r["id"] for r in REQUESTS]
_new_name = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
RUN_UI = dict(
  experiment=widgets.Combobox(value=_new_name, options=experiments(), ensure_option=False,
                              description="Experiment", placeholder="a new name, or pick one to continue",
                              layout=widgets.Layout(width="420px")),
  options={o: widgets.Checkbox(value=True, description=o, indent=False)
           for o in ("1_keyframes", "2_hybrid", "3_unimate")},
  requests=widgets.SelectMultiple(options=_ids, value=tuple(_ids), rows=min(12, len(_ids)), description="Requests"),
  repeats=widgets.BoundedIntText(value=2, min=1, max=10, description="Repeats"),
)
_display(widgets.VBox([
  RUN_UI["experiment"],
  widgets.HBox([widgets.Label("Options:")] + list(RUN_UI["options"].values())),
  RUN_UI["requests"], RUN_UI["repeats"],
  widgets.HTML("<small>Ctrl/Shift-click to pick several requests. Then run the next cell.</small>"),
]))

In [ ]:
# @title Run the chosen experiment (~1 min per request per option per repeat)
# Saves after every run to runs/<experiment>/results.pkl; rerun the cell to continue
# where it stopped. Also saves the settings and the exact LLM prompts next to it.
import random

EXP = re.sub(r"[^\w.-]+", "_", RUN_UI["experiment"].value.strip()) or _new_name
OPTIONS = [o for o, cb in RUN_UI["options"].items() if cb.value]
ONLY = list(RUN_UI["requests"].value)
REPEATS = int(RUN_UI["repeats"].value)
if not OPTIONS or not ONLY:
  raise ValueError("pick at least one option and one request above")

exp_dir = RUNS / EXP
exp_dir.mkdir(parents=True, exist_ok=True)
_settings = dict(config=dataclasses.asdict(CFG), llm=f"{llm.backend}:{llm.model}")
_cfg_file = exp_dir / "config.json"
_old = json.loads(_cfg_file.read_text()) if _cfg_file.exists() else {}
for k in ("config", "llm"):
  if k in _old and _old[k] != _settings[k]:
    diff = ({kk: (_old[k].get(kk), v) for kk, v in _settings[k].items() if _old[k].get(kk) != v}
            if isinstance(_settings[k], dict) else (_old[k], _settings[k]))
    print(f"WARNING: {EXP} was started with different {k}: {diff}. New runs use the current ones.")
_cfg_file.write_text(json.dumps(dict(
  created=_old.get("created", datetime.datetime.now().isoformat(timespec="seconds")),
  options=sorted(set(_old.get("options", [])) | set(OPTIONS)),
  requests=sorted(set(_old.get("requests", [])) | set(ONLY)),
  repeats=max(_old.get("repeats", 0), REPEATS), **_settings), indent=1))
_prompts = dict(keyframes=KEYFRAME_SYSTEM, hybrid=HYBRID_SYSTEM, text=TEXT_SYSTEM)
_pdir = exp_dir / "prompts"
_pdir.mkdir(exist_ok=True)
for k, v in _prompts.items():  # keep every version of the prompts this experiment ran with
  existing = sorted(_pdir.glob(f"{k}*.txt"))
  if not existing or existing[-1].read_text() != v:
    (_pdir / f"{k}_{datetime.datetime.now():%Y%m%d-%H%M%S}.txt").write_text(v)

run_results = load_experiment(EXP)
done = {(r["request_id"], r["option"], r.get("rep", 0)) for r in run_results}
todo = []
for rep in range(REPEATS):
  for req in REQUESTS:
    if req["id"] not in ONLY:
      continue
    opts = list(OPTIONS)
    random.Random(f"{req['id']}/{rep}").shuffle(opts)  # no option always goes first
    todo += [(req, opt, rep) for opt in opts if (req["id"], opt, rep) not in done]

print(f"experiment {EXP}: {len(todo)} runs to do, {len(run_results)} already done | LLM: {llm.backend} {llm.model}")
print("saving after every run to", exp_dir / "results.pkl")
for n, (req, opt, rep) in enumerate(todo, 1):
  cfg = dataclasses.replace(CFG, seed=CFG.seed + 7919 * rep)
  rec = run_one(opt, req, cfg)
  rec.update(rep=rep, experiment=EXP, config=dataclasses.asdict(cfg), llm_model=f"{llm.backend}:{llm.model}")
  run_results.append(rec)
  save_experiment(EXP, run_results)
  st = " ".join(f"{k} {v:.1f}" for k, v in rec["latency"].items() if v >= 0.05)
  sp = rec.get("spec")
  print(f"[{n}/{len(todo)}] {req['id']:10} {opt:11} rep {rep}: {'PASS' if rec['ok'] else 'fail'}"
        f" {rec['total_s']:5.1f} s ({st})" + (f" | spec {'ok' if sp['ok'] else 'off'}: {sp['measured']}" if sp else "")
        + ("" if rec["ok"] else f" | {rec['reasons'][0][:90]}"))
print(f"done: {len(run_results)} results in experiment {EXP}. Pick it in section 8 to look at it.")

## 8. Answers 1 and 2: latency and objective quality
First pick the experiments to look at. Everything after this (the numbers, the viewer, videos, ratings, the
diagnostic) uses that choice. Several are merged; when two have the same request, option and repeat, the newer one is
used. These cells need only Setup (and Core, for the diagnostic), so they also work on a cheap CPU runtime.

How to read it:

- **Latency:** the median is the typical wait; **p90** is what a demo audience will sometimes sit through. Retries
  (option 1's verifier loop; new seeds in 2 and 3) show up in p90.
- **Quality, objective:**
  - `passed`: how often the motion is usable at all.
  - `spec_ok`: on the precise requests, did it do what was asked (the right paw, about the right height and time)?
  - `slide`, `floating`, `com_outside`: physical plausibility.
  - `jerk_rms`: smoothness; lower is smoother, but a motion that does nothing is perfectly smooth.
  - `joint_travel`, `tilt_range`: how much it actually moves, so read them next to `jerk_rms`.
- **What the numbers can't tell you:** whether it looks like a play bow, or like a living thing. That is what the
  blind ratings (sections 9 and 10) are for.

In [ ]:
# @title Choose which experiments to look at (sections 8-10 use these)
# Pick one or several; they are merged. When two have the same request, option and
# repeat, the newer experiment's result is used: rerun just one option and compare
# it with the others' older results. Needs only the Setup cell.
from collections import Counter

import ipywidgets as widgets
from IPython.display import display as _display

_exps = experiments()  # oldest first
ACTIVE_UI = widgets.SelectMultiple(options=_exps, value=tuple(_exps), rows=min(8, max(2, len(_exps))),
                                   description="Experiments", layout=widgets.Layout(width="420px"))
_active_status = widgets.HTML()


def _activate(_change=None):
  global results
  names = [e for e in _exps if e in ACTIVE_UI.value]  # oldest -> newest, so newer wins
  results = merge_experiments(names)
  _atomic_pickle(results, ACTIVE_FILE)
  by = Counter((r["experiment"], r["option"]) for r in results)
  lines = [f"<b>{len(results)} results</b> (in {ACTIVE_FILE}):"]
  for e in names:
    parts = [f"{o} x{n}" for (ee, o), n in sorted(by.items()) if ee == e]
    lines.append(f"&nbsp; {e}: {', '.join(parts) if parts else 'nothing used (all replaced by newer experiments)'}")
  _active_status.value = "<br>".join(lines)


ACTIVE_UI.observe(_activate, names="value")
_activate()
_display(widgets.VBox([ACTIVE_UI, _active_status,
                       widgets.HTML("<small>Ctrl/Shift-click to pick several. The cells below read the selection"
                                    " when they run.</small>")]))

In [ ]:
# @title Results: latency and objective quality
import matplotlib.pyplot as plt
import pandas as pd

rows = []
for r in results:
  m = r.get("metrics") or {}
  sp = r.get("spec") or {}
  ka = r.get("kf_adherence") or {}
  row = dict(experiment=r.get("experiment", ""), request=r["request_id"], kind=r["kind"], option=r["option"], rep=r.get("rep", 0),
             passed=r["ok"], total_s=r["total_s"], llm_calls=len(r["llm_calls"]),
             spec_ok=sp.get("ok", np.nan), spec_error=sp.get("error", np.nan),
             kf_foot_cm_max=ka.get("foot_cm_max", np.nan),
             n_keyframes=r.get("n_keyframes", len(r.get("keyframes") or []) or np.nan))
  row.update({f"t_{k}": v for k, v in r["latency"].items()})
  row.update(m)
  rows.append(row)
df = pd.DataFrame(rows)
df.to_csv(RESULTS_ROOT / "latest_summary.csv", index=False)  # the current selection, as a table
stage_cols = [c for c in ["t_llm", "t_build", "t_verify", "t_guide", "t_unimate", "t_rank", "t_fit", "t_finalize", "t_gate"] if c in df]

print("=" * 80, "\nQ2. LATENCY: request -> motion ready (s). Excludes rate-limit waits and the robot's ~1.5 s controller start.")
lat = df.groupby("option").agg(
  median=("total_s", "median"), p90=("total_s", lambda s: s.quantile(0.9)), max=("total_s", "max"),
  llm_calls=("llm_calls", "mean"))
display(lat.round(2))
print("Median seconds per stage (a stage that didn't run counts as 0):")
display(df.groupby("option")[stage_cols].apply(lambda g: g.fillna(0).median()).round(2))
print("Median latency by request kind:")
display(df.pivot_table(index="kind", columns="option", values="total_s", aggfunc="median").round(1))

print("=" * 80, "\nQ1. QUALITY (objective). Your blind ratings are in section 10.")
q = df.groupby("option").agg(
  passed=("passed", "mean"), spec_ok=("spec_ok", "mean"), spec_error=("spec_error", "median"),
  holds_up=("physics", lambda s: (s == "HOLDS UP").mean()), falls=("physics", lambda s: (s == "FALLS").mean()),
  slide_worst_cm=("slide_worst_cm", "median"), floating=("floating_frac", "median"),
  com_outside=("com_outside_frac", "median"), jerk_rms=("jerk_rms", "median"),
  joint_travel=("joint_travel_rad", "median"), tilt_range=("body_tilt_range_deg", "median"),
  duration=("duration_s", "median"), keyframes=("n_keyframes", "median"),
  kf_foot_cm_max=("kf_foot_cm_max", "median"))
display(q.round(3))
print("passed (gate) = no fall, balanced, no slide, within speed/limits | spec_ok = precise requests that"
      " matched their numbers | jerk_rms lower = smoother | joint_travel/tilt_range higher = more motion")
print("\nPass rate by request kind:")
display(df.pivot_table(index="kind", columns="option", values="passed", aggfunc="mean").round(2))
print("Per request (pass rate over repeats | spec):")
display(df.pivot_table(index="request", columns="option", values=["passed", "spec_ok"], aggfunc="mean").round(2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
opts = sorted(df.option.unique())
ax[0].boxplot([df[df.option == o].total_s.dropna() for o in opts])
ax[0].set_xticks(range(1, len(opts) + 1), opts)
ax[0].set_ylabel("seconds, request -> motion ready")
ax[0].set_title("Latency")
med = df.groupby("option")[stage_cols].apply(lambda g: g.fillna(0).median())
bottom = np.zeros(len(med))
for c in stage_cols:
  ax[1].bar(med.index, med[c], bottom=bottom, label=c[2:])
  bottom += med[c].values
ax[1].set_title("Median time per stage")
ax[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 9. Look at them, and rate them blind
Like the labs: the next cell writes the viewer program, and the one after runs it with viser's share link
(`--share`). Open the link it prints. The cell keeps running while you watch; stop it to close the viewer. The
link is public for 24 hours, so only send it to people you want rating. Both cells need only the Setup cell and
the saved results, so they also work after a runtime restart.

- **Request / Repeat:** what to show. **Blind:** shuffled order, grey robots, no names. Turn it off afterwards to see
  which option was which, with its numbers and why a failed one was rejected.
- **Show → physics replay:** the motion as plain physics plays it (does it tip over?). **UniMate rig:** the skeleton
  UniMate generated, under the fitted robot.
- **Rate:** "matches the request" and "looks natural", 1-5, and whether you'd show it on demo day. **Save ratings**
  appends to `ratings.csv`.

**No viewer?** "Videos and ratings right here" renders the same three motions side by side (blind, same order)
with the same rating form, saved to the same `ratings.csv`. It renders in a separate process, so a GPU that can't
render makes the cell say so instead of crashing the runtime. MuJoCo's off-screen renderer needs graphics support,
which a G4 has and an A100 may not.

In [ ]:
%%writefile /content/bench_viewer.py
"""The three options side by side in viser, with blind ratings.

Runs as its own program, like the labs' `play_motion --viewer viser --share True`:
the notebook cell keeps running while you watch, and stopping the cell stops the
viewer. It reads the selected results (/content/active_results.pkl) and appends
ratings to ratings.csv. Three Puppers (A, B, C) do the same request; in blind mode the
order is shuffled per request and all three are grey, so you judge the motion,
not the label.

  python bench_viewer.py RESULTS.pkl RATINGS.csv [--share]
"""

import argparse
import csv
import datetime
import pickle
import sys
import threading
import time
import zlib
from pathlib import Path
from time import perf_counter

import mujoco
import numpy as np

sys.path.insert(0, "/content/pupper-mjlab/src")
from mjlab.tasks.tracking.config.pupper import motion_design as md  # noqa: E402
from mjlab.tasks.tracking.config.pupper import unimate as U  # noqa: E402

import viser  # noqa: E402

FPS = 50.0
KIN = md.kinematics()
SK = U.SKELETON

VIEW_PORT = 8765  # Colab's runtime uses 8080; viser moves to the next free port if this one is taken
OPTION_COLORS = {"1_keyframes": (80, 130, 230), "2_hybrid": (235, 145, 50), "3_unimate": (70, 175, 110)}
GREY = (175, 175, 180)
SLOT_Y = (-0.45, 0.0, 0.45)  # A, B, C from left to right, seen from the default camera
SLOT_NAMES = ("A", "B", "C")
RIG_EDGES = [(p, j) for j, p in enumerate(SK["parents"]) if p >= 0]


MESH_CELL_M = 0.006  # simplify meshes to a 6 mm grid: ~8 MB of meshes -> under 1 MB to send


def _simplify(v, f, cell):
  """Vertex clustering: merge vertices in the same grid cell, drop collapsed triangles."""
  key = np.floor(v / cell).astype(np.int64)
  _, inv = np.unique(key, axis=0, return_inverse=True)
  inv = inv.reshape(-1)
  cnt = np.bincount(inv)
  nv = np.zeros((len(cnt), 3))
  np.add.at(nv, inv, v)
  nv /= cnt[:, None]
  nf = inv[f]
  keep = (nf[:, 0] != nf[:, 1]) & (nf[:, 1] != nf[:, 2]) & (nf[:, 0] != nf[:, 2])
  return nv.astype(np.float32), nf[keep].astype(np.int32)


def _visual_geoms(model, cell=MESH_CELL_M):
  out = []
  for g in range(model.ngeom):
    if model.geom_type[g] != mujoco.mjtGeom.mjGEOM_MESH or model.geom_group[g] == 3:
      continue  # group 3: collision hulls
    mid = model.geom_dataid[g]
    v = model.mesh_vert[model.mesh_vertadr[mid]: model.mesh_vertadr[mid] + model.mesh_vertnum[mid]]
    f = model.mesh_face[model.mesh_faceadr[mid]: model.mesh_faceadr[mid] + model.mesh_facenum[mid]]
    out.append((g, *_simplify(v.astype(np.float64), f, cell)))
  return out


class Viewer:
  def __init__(self, results, ratings_path, port=VIEW_PORT, options=("1_keyframes", "2_hybrid", "3_unimate")):
    self.options = list(options)
    self.ratings_path = Path(ratings_path)
    self.by_req: dict = {}
    for r in results:
      if r.get("motion") is not None:
        self.by_req.setdefault(r["request_id"], {}).setdefault(r["option"], {})[r.get("rep", 0)] = r
    if not self.by_req:
      raise SystemExit("no results with a motion yet: run section 7 first")
    self.texts = {r["request_id"]: r["request"] for r in results}
    self.model = KIN.model
    self.data = mujoco.MjData(self.model)
    self.t = 0.0
    self.alive = True
    s = self.server = viser.ViserServer(port=port, verbose=False)
    s.initial_camera.position = (1.15, 0.22, 0.45)  # shifted right: the GUI panel covers the right side
    s.initial_camera.look_at = (0.0, 0.22, 0.08)
    s.scene.add_grid("/grid", width=2.0, height=1.6, cell_size=0.1, section_size=0.5)
    geoms = _visual_geoms(self.model)
    self.slots = []
    for k, y in enumerate(SLOT_Y):
      s.scene.add_frame(f"/slot{k}", show_axes=False, position=(0.0, y, 0.0))
      meshes = [(g, s.scene.add_mesh_simple(f"/slot{k}/g{g}", v, f, color=GREY, side="double"))
                for g, v, f in geoms]
      label = s.scene.add_label(f"/slot{k}/label", SLOT_NAMES[k], position=(0.0, 0.0, 0.32))
      skel = s.scene.add_line_segments(f"/slot{k}/rig", np.zeros((len(RIG_EDGES), 2, 3), np.float32),
                                       (230, 60, 60), thickness=0.004, visible=False)
      self.slots.append(dict(meshes=meshes, label=label, skel=skel, rec=None, option=None))

    ids = list(self.by_req)
    g = s.gui
    self.req_dd = g.add_dropdown("Request", [f"{i}: {self.texts[i][:48]}" for i in ids], initial_value=f"{ids[0]}: {self.texts[ids[0]][:48]}")
    reps = sorted({rep for d in self.by_req.values() for o in d.values() for rep in o})
    self.rep_dd = g.add_dropdown("Repeat", [str(r) for r in reps], initial_value=str(reps[0]))
    self.blind = g.add_checkbox("Blind (shuffle, hide names)", True)
    self.mode = g.add_dropdown("Show", ["reference motion", "physics replay"], initial_value="reference motion")
    self.show_rig = g.add_checkbox("UniMate rig (red lines)", False)
    self.playing = g.add_checkbox("Play", True)
    self.loop = g.add_checkbox("Loop", True)
    self.speed = g.add_slider("Speed", 0.1, 2.0, 0.05, 1.0)
    self.time = g.add_slider("Time (s)", 0.0, 10.0, 0.01, 0.0)
    self.info = g.add_markdown("")
    with g.add_folder("Rate (1 = poor, 5 = great)"):
      self.rater = g.add_text("Your name", "")
      self.rate = {}
      for name in SLOT_NAMES:
        self.rate[name] = (
          g.add_slider(f"{name}: matches the request", 1, 5, 1, 3),
          g.add_slider(f"{name}: looks natural", 1, 5, 1, 3),
          g.add_checkbox(f"{name}: I'd show this on demo day", False),
        )
      save = g.add_button("Save ratings")
      self.status = g.add_markdown("")

    @self.req_dd.on_update
    def _(_e):
      self._load()

    @self.rep_dd.on_update
    def _(_e):
      self._load()

    @self.blind.on_update
    def _(_e):
      self._load()

    @self.time.on_update
    def _(_e):
      if not self.playing.value:
        self.t = float(self.time.value)

    @save.on_click
    def _(_e):
      self._save()

    self._load()
    threading.Thread(target=self._run, daemon=True).start()

  # ---------------------------------------------------------------- state
  def _rid(self):
    return self.req_dd.value.split(":")[0]

  def _load(self):
    self._version = getattr(self, "_version", 0) + 1  # forces a redraw
    rid, rep = self._rid(), int(self.rep_dd.value)
    order = list(self.options)
    if self.blind.value:
      np.random.default_rng(zlib.crc32(f"{rid}/{rep}".encode())).shuffle(order)
    lines = [f"**{self.texts[rid]}**", ""]
    self.T = 0.0
    for k, opt in enumerate(order):
      sl = self.slots[k]
      rec = self.by_req[rid].get(opt, {}).get(rep)
      sl["rec"], sl["option"] = rec, opt
      color = GREY if self.blind.value else OPTION_COLORS.get(opt, GREY)
      for _g, h in sl["meshes"]:
        h.color = color
        h.visible = rec is not None
      sl["label"].text = SLOT_NAMES[k] if self.blind.value else f"{SLOT_NAMES[k]}: {opt}"
      if rec is None:
        lines.append(f"- {SLOT_NAMES[k]}: (no result)")
        continue
      self.T = max(self.T, (len(rec["motion"]["root_pos"]) - 1) / FPS)
      if not self.blind.value:
        m = rec["metrics"] or {}
        sp = rec.get("spec")
        lines.append(
          f"- **{SLOT_NAMES[k]} = {opt}**: {'PASS' if rec['ok'] else 'fail'}, {rec['total_s']:.1f} s, "
          f"{m.get('physics')}, slide {m.get('slide_worst_cm')} cm"
          + (f", spec {'ok' if sp['ok'] else 'off'} ({sp['measured']})" if sp else "")
          + (f", {rec.get('n_keyframes', len(rec.get('keyframes') or []))} keyframes" if rec.get("keyframes") else "")
          + (f", UniMate: {' / '.join(rec['unimate_prompts'])}" if rec.get("unimate_prompts") else "")
        )
        if not rec["ok"]:
          lines.append(f"  - rejected: {rec['reasons'][0][:160]}")
    self.time.max = max(0.1, round(self.T, 2))
    self.t = 0.0
    self.info.content = "\n".join(lines)

  def _save(self):
    rid, rep = self._rid(), int(self.rep_dd.value)
    path = self.ratings_path
    new = not path.exists()
    with path.open("a", newline="") as f:
      w = csv.writer(f)
      if new:
        w.writerow(["experiment", "time", "rater", "request_id", "rep", "slot", "option", "blind", "match", "natural", "demo"])
      for k, name in enumerate(SLOT_NAMES):
        if self.slots[k]["rec"] is None:
          continue
        a, b, c = self.rate[name]
        w.writerow([self.slots[k]["rec"].get("experiment", ""), datetime.datetime.now().isoformat(timespec="seconds"), self.rater.value, rid, rep,
                    name, self.slots[k]["option"], self.blind.value, a.value, b.value, int(c.value)])
    self.status.content = f"Saved ratings for **{rid}** (repeat {rep}) to `{path}`."

  # ---------------------------------------------------------------- drawing
  def _pose(self, rec, i):
    if self.mode.value == "physics replay" and rec.get("replay") is not None:
      q = rec["replay"][min(i, len(rec["replay"]) - 1)]
      return q[:3], q[3:7], q[7:]
    m = rec["motion"]
    i = min(i, len(m["root_pos"]) - 1)
    return m["root_pos"][i], m["root_quat"][i], m["joint_pos"][i]

  def _draw(self):
    i = int(self.t * FPS)
    quat = np.zeros(4)
    with self.server.atomic():
      for sl in self.slots:
        rec = sl["rec"]
        if rec is None:
          continue
        rp, rq, jq = self._pose(rec, i)
        self.data.qpos[:3], self.data.qpos[3:7] = rp, rq
        self.data.qpos[KIN.qadr] = jq
        mujoco.mj_kinematics(self.model, self.data)
        for g, h in sl["meshes"]:
          mujoco.mju_mat2Quat(quat, self.data.geom_xmat[g])
          h.position = self.data.geom_xpos[g].copy()
          h.wxyz = quat.copy()
        sk = rec.get("skeleton")
        show = self.show_rig.value and sk is not None
        if show:
          j = int(round((i - rec.get("pre_frames", 0)) / rec.get("slow", 1.0)))
          show = 0 <= j < len(sk)
        if show:
          p = sk[j]
          sl["skel"].points = np.array([[p[a], p[b]] for a, b in RIG_EDGES], np.float32)
        sl["skel"].visible = bool(show)

  def _run(self):
    """20 frames a second, and only what changed: through Colab's proxy or the
    share relay every pose update is a message, so nothing is sent while no
    browser is connected or nothing moves."""
    last, last_slider, drawn = perf_counter(), 0.0, None
    while self.alive:
      now = perf_counter()
      if self.playing.value:
        self.t += (now - last) * self.speed.value
        if self.t > self.T + 0.5:  # a short pause at the end
          self.t = 0.0 if self.loop.value else self.T
      last = now
      if not self.server.get_clients():
        drawn = None  # draw everything again when someone connects
        time.sleep(0.2)
        continue
      if self.playing.value and now - last_slider > 0.25:
        self.time.value = round(min(self.t, self.T), 2)
        last_slider = now
      state = (int(self.t * FPS), self._version, self.mode.value, self.show_rig.value)
      if state != drawn:
        try:
          self._draw()
          drawn = state
        except Exception as e:  # noqa: BLE001
          print("viewer:", e)
      time.sleep(1 / 20)

  def close(self):
    self.alive = False
    self.server.stop()


def main():
  ap = argparse.ArgumentParser()
  ap.add_argument("results")
  ap.add_argument("ratings")
  ap.add_argument("--port", type=int, default=VIEW_PORT)
  ap.add_argument("--share", action="store_true")
  a = ap.parse_args()
  results = pickle.loads(Path(a.results).read_bytes())
  v = Viewer(results, a.ratings, port=a.port)
  print(f"Viewer running on port {v.server.get_port()} ({len(v.by_req)} requests). Stop the cell to close it.",
        flush=True)
  if a.share:
    print("Open the viewer:", v.server.request_share_url(), flush=True)
  try:
    while True:
      time.sleep(1.0)
  except KeyboardInterrupt:
    pass
  finally:
    v.close()


if __name__ == "__main__":
  main()

In [ ]:
# @title Open the viewer (the cell keeps running; stop it to close the viewer)
# As in the labs: viser runs as its own program and prints a share link. The link
# is public (anyone with it can see the scene) and lasts 24 hours.
# Shows the experiments picked in section 8; ratings go to ratings.csv.
if not ACTIVE_FILE.exists():
  raise FileNotFoundError("pick the experiments to look at first (section 8, the first cell)")
_results_file, _ratings_file = ACTIVE_FILE, RATINGS_FILE
!python /content/bench_viewer.py "{_results_file}" "{_ratings_file}" --share

In [ ]:
%%writefile /content/bench_render.py
"""Render one request's motions side by side to an mp4, in its own process.

MuJoCo's off-screen renderer is native code; if the GPU's graphics driver
can't do it (e.g. EGL on a compute-only runtime), it crashes the process. Run
here, that kills only this program, not the notebook.

  python bench_render.py RESULTS.pkl REQUEST_ID REP OUT.mp4 --options a,b,c --labels A,B,C [--physics]
"""

import argparse
import pickle
import sys
import types
from pathlib import Path

import numpy as np

sys.path.insert(0, "/content/pupper-mjlab/src")
import mediapy  # noqa: E402
from mjlab.tasks.tracking.config.pupper import motion_design as md  # noqa: E402
from mjlab.tasks.tracking.config.pupper import unimate as U  # noqa: E402


def main():
  ap = argparse.ArgumentParser()
  ap.add_argument("results")
  ap.add_argument("request")
  ap.add_argument("rep", type=int)
  ap.add_argument("out")
  ap.add_argument("--options", required=True)
  ap.add_argument("--labels", required=True)
  ap.add_argument("--physics", action="store_true")
  a = ap.parse_args()
  results = pickle.loads(Path(a.results).read_bytes())
  recs = {r["option"]: r for r in results
          if r["request_id"] == a.request and r.get("rep", 0) == a.rep and r.get("motion") is not None}
  fits = []
  for opt in a.options.split(","):
    r = recs[opt]
    if a.physics and r.get("replay") is not None:
      q = np.asarray(r["replay"], float)
      m = md.Motion(fps=50.0, root_pos=q[:, :3], root_quat=q[:, 3:7], joint_pos=q[:, 7:])
    else:
      m = md.Motion(fps=50.0, root_pos=np.asarray(r["motion"]["root_pos"]),
                    root_quat=np.asarray(r["motion"]["root_quat"]), joint_pos=np.asarray(r["motion"]["joint_pos"]))
    fits.append(types.SimpleNamespace(motion=m, skeleton=None))
  video = U.render(fits, a.labels.split(","), width=360, height=300, skeleton=False)
  mediapy.write_video(a.out, video, fps=50)
  print("wrote", a.out)


if __name__ == "__main__":
  main()

In [ ]:
# @title (If the viewer won't connect) Videos and ratings right here
# The same comparison without a server: the three motions rendered side by side
# (blind: same shuffled order as the viewer), and the same ratings form, saved to
# the same ratings.csv. Pick a request, run the cell, rate, press Save; repeat.
# Rendering runs in its own process: if this GPU can't render, the cell says so
# instead of crashing the runtime. Needs the Setup cell and section 8's experiment choice.
import csv
import datetime
import os
import pickle
import subprocess
import sys
import zlib
from pathlib import Path

import ipywidgets as widgets
import numpy as np
from IPython.display import Video
from IPython.display import display as _display

VIDEO_REQUEST = "paw"  # @param {type:"string"}
VIDEO_REP = 0  # @param {type:"integer"}
VIDEO_BLIND = True  # @param {type:"boolean"}
VIDEO_PHYSICS = False  # @param {type:"boolean"}
# VIDEO_PHYSICS: show the physics replay instead of the reference motion.

SLOT_NAMES = ("A", "B", "C")
RESULTS_FILE = ACTIVE_FILE  # the experiments picked in section 8
_all = pickle.loads(RESULTS_FILE.read_bytes())
recs = {r["option"]: r for r in _all
        if r["request_id"] == VIDEO_REQUEST and r.get("rep", 0) == VIDEO_REP and r.get("motion") is not None}
order = sorted(recs)
if VIDEO_BLIND:
  np.random.default_rng(zlib.crc32(f"{VIDEO_REQUEST}/{VIDEO_REP}".encode())).shuffle(order)  # same order as the viewer

if not recs:
  print("no results for that request and repeat; ids:", sorted({r["request_id"] for r in _all}))
else:
  print(next(iter(recs.values()))["request"])
  labels = list(SLOT_NAMES[: len(order)]) if VIDEO_BLIND else [f"{s}: {o}" for s, o in zip(SLOT_NAMES, order)]
  out = Path("/content") / f"video_{VIDEO_REQUEST}_{VIDEO_REP}{'_physics' if VIDEO_PHYSICS else ''}.mp4"
  done = None
  for gl in ("egl", "osmesa"):
    cmd = [sys.executable, "/content/bench_render.py", str(RESULTS_FILE), VIDEO_REQUEST, str(VIDEO_REP),
           str(out), "--options", ",".join(order), "--labels", ",".join(labels)] + (["--physics"] if VIDEO_PHYSICS else [])
    p = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "MUJOCO_GL": gl})
    if p.returncode == 0 and out.exists():
      done = gl
      break
    print(f"rendering with MUJOCO_GL={gl} failed (exit {p.returncode}): {(p.stderr or p.stdout).strip()[-300:]}")
  if done:
    _display(Video(str(out), embed=True, width=1080))
  else:
    print("This runtime can't render MuJoCo off-screen. Use the viewer, or a G4 runtime.")

  rater = widgets.Text(description="Your name")
  form = {}
  rows_ui = [rater]
  for s in SLOT_NAMES[: len(order)]:
    form[s] = (widgets.IntSlider(3, 1, 5, description=f"{s} matches"),
               widgets.IntSlider(3, 1, 5, description=f"{s} natural"),
               widgets.Checkbox(False, description=f"{s}: demo-worthy"))
    rows_ui.append(widgets.HBox(list(form[s])))
  save_btn = widgets.Button(description="Save ratings", button_style="primary")
  status = widgets.HTML()

  def _save_video_ratings(_b):
    rows = []
    for s, o in zip(SLOT_NAMES, order):
      a, b, c = form[s]
      rows.append([recs[o].get("experiment", ""), datetime.datetime.now().isoformat(timespec="seconds"), rater.value,
                   VIDEO_REQUEST, VIDEO_REP, s, o, VIDEO_BLIND, a.value, b.value, int(c.value)])
    append_ratings(rows)
    status.value = f"Saved ratings for <b>{VIDEO_REQUEST}</b> (repeat {VIDEO_REP}) to {RATINGS_FILE}."

  save_btn.on_click(_save_video_ratings)
  _display(widgets.VBox(rows_ui + [save_btn, status]))

In [ ]:
# @title Export videos of every request (to share; nothing has to keep running)
# One mp4 per request (the three options side by side) and an index.html page with
# the request, the videos and the key numbers, saved to exports/<time>/ and
# downloaded as a zip. Open index.html in any browser, offline.
# Rendering runs in a separate process (bench_render.py: run the renderer cell above
# first). It tries the GPU renderer (EGL), then the CPU one (OSMesa), so it also works
# on a cheap CPU-only runtime: Setup, the renderer cell, then this cell.
import datetime
import html
import os
import pickle
import shutil
import subprocess
import sys
import zlib
from pathlib import Path

import numpy as np

EXPORT_REP = 0  # @param {type:"integer"}
EXPORT_NAMED = True  # @param {type:"boolean"}
# EXPORT_NAMED: label each panel with its option. Off: blind A/B/C in the viewer's order.

RESULTS_FILE = ACTIVE_FILE  # the experiments picked in section 8
if not Path("/content/bench_render.py").exists():
  raise FileNotFoundError("run the renderer program cell (%%writefile /content/bench_render.py) first")
res = pickle.loads(RESULTS_FILE.read_bytes())
out_dir = RESULTS_ROOT / "exports" / datetime.datetime.now().strftime("%Y-%m-%d_%H%M%S")
out_dir.mkdir(parents=True, exist_ok=True)
if subprocess.run("ldconfig -p | grep -q libOSMesa", shell=True).returncode != 0:
  print("installing the CPU renderer (OSMesa)...")
  subprocess.run("apt-get -qq update && apt-get -qq install -y libosmesa6 > /dev/null", shell=True)

gls = ["egl", "osmesa"]
rows = []
for rid in sorted({r["request_id"] for r in res}):
  recs = {r["option"]: r for r in res
          if r["request_id"] == rid and r.get("rep", 0) == EXPORT_REP and r.get("motion") is not None}
  if not recs:
    continue
  order = sorted(recs)
  if not EXPORT_NAMED:
    np.random.default_rng(zlib.crc32(f"{rid}/{EXPORT_REP}".encode())).shuffle(order)
  labels = ([f"{o} {'PASS' if recs[o]['ok'] else 'fail'} {recs[o]['total_s']:.1f}s" for o in order]
            if EXPORT_NAMED else ["A", "B", "C"][: len(order)])
  mp4 = out_dir / f"{rid}_rep{EXPORT_REP}{'' if EXPORT_NAMED else '_blind'}.mp4"
  for gl in list(gls):
    p = subprocess.run([sys.executable, "/content/bench_render.py", str(RESULTS_FILE), rid, str(EXPORT_REP), str(mp4),
                        "--options", ",".join(order), "--labels", ",".join(labels)],
                       capture_output=True, text=True, env={**os.environ, "MUJOCO_GL": gl})
    if p.returncode == 0 and mp4.exists():
      gls = [gl]  # keep using the renderer that works
      break
    print(f"  {rid}: MUJOCO_GL={gl} failed: {(p.stderr or p.stdout).strip()[-200:]}")
  else:
    print(f"  {rid}: could not render")
    continue
  print(f"{rid}: {mp4.name} ({gls[0]})")
  rows.append((rid, next(iter(recs.values()))["request"], mp4.name, [(o, recs[o]) for o in order]))

# A page to open in a browser: each request, its video, and the numbers behind it.
parts = ["<!doctype html><meta charset='utf-8'><title>Pupper generation benchmark</title>",
         "<style>body{font-family:sans-serif;max-width:1150px;margin:24px auto;padding:0 16px}"
         "video{width:100%;background:#eee}td,th{padding:2px 10px;text-align:left}</style>",
         f"<h1>Pupper: keyframes vs UniMate</h1><p>Repeat {EXPORT_REP}. Panels left to right as labelled."]
for rid, text, name, opts in rows:
  parts.append(f"<h2>{html.escape(rid)}: {html.escape(text)}</h2><video src='{name}' controls loop muted></video>")
  if EXPORT_NAMED:
    parts.append("<table><tr><th>option</th><th>gate</th><th>latency</th><th>physics</th><th>precision</th><th>notes</th></tr>")
    for o, r in opts:
      m, sp = r.get("metrics") or {}, r.get("spec")
      note = " / ".join(r.get("unimate_prompts") or []) or (f"{len(r.get('keyframes') or [])} keyframes" if r.get("keyframes") else "")
      if not r["ok"]:
        note = "rejected: " + r["reasons"][0][:120]
      parts.append(f"<tr><td>{o}</td><td>{'PASS' if r['ok'] else 'fail'}</td><td>{r['total_s']:.1f} s</td>"
                   f"<td>{m.get('physics', '')}</td><td>{html.escape(sp['measured']) if sp else ''}</td>"
                   f"<td>{html.escape(note)}</td></tr>")
    parts.append("</table>")
(out_dir / "index.html").write_text("\n".join(parts), encoding="utf-8")

zip_path = shutil.make_archive(f"/content/pupper_videos_{out_dir.name}", "zip", out_dir)
print(f"\n{len(rows)} videos + index.html in {out_dir}; zip: {zip_path}")
try:
  from google.colab import files

  files.download(zip_path)
except Exception as e:  # noqa: BLE001
  print("download it from the Files pane:", e)

## 10. Your ratings
Blind ratings only, when there are any. Rate at least a few requests of each kind first.

In [ ]:
# @title Human ratings (from the viewer's and the video cell's Save buttons)
# Ratings of the experiments picked in section 8 only. Needs only the Setup cell and that choice.
import pickle

import pandas as pd

if not RATINGS_FILE.exists():
  print("No ratings yet: rate some requests in the viewer (section 9) first.")
else:
  _act = pickle.loads(ACTIVE_FILE.read_bytes())
  _shown = {(r["experiment"], r["request_id"], r["option"], r.get("rep", 0)) for r in _act}
  kinds = {r["request_id"]: r["kind"] for r in _act}
  rt = pd.read_csv(RATINGS_FILE).drop_duplicates(["experiment", "rater", "request_id", "rep", "slot"], keep="last")
  rt = rt[[(e, q, o, int(p)) in _shown for e, q, o, p in zip(rt.experiment, rt.request_id, rt.option, rt.rep)]]
  if rt.empty:
    print("No ratings for the selected experiments yet.")
  else:
    blind = rt[rt.blind.astype(str) == "True"]
    print(f"{len(rt)} ratings ({len(blind)} blind) from {rt.rater.nunique()} rater(s)")
    use = (blind if len(blind) else rt).assign(kind=lambda x: x.request_id.map(kinds))
    print("Mean rating (1-5) and how often it was demo-worthy, blind ratings only if there are any:")
    display(use.groupby("option")[["match", "natural", "demo"]].mean().round(2))
    print("By request kind:")
    display(use.pivot_table(index="kind", columns="option", values=["match", "natural"], aggfunc="mean").round(2))
    wins = use.assign(score=use.match + use.natural).sort_values("score").groupby(["rater", "request_id", "rep"]).tail(1)
    print("How often each option was rated best for a request (match + natural):")
    display(wins.option.value_counts(normalize=True).round(2))

## 10b. Diagnose UniMate: generation, retargeting, or selection?
Takes every UniMate result in `results.pkl` and measures how much it moves at each stage: UniMate's own output,
after the fit's 40% shrink of horizontal motion, after the IK fit to Pupper, and which of the 4 samples was kept.
Where the amplitude drops tells you whether to fix the generator, the retargeting (e.g. a GMR-style fit), or the
sample ranking. Needs only Setup and the Core cell, so it runs on a CPU runtime.

In [ ]:
# @title Diagnose: where does UniMate's motion get lost? (CPU is fine, ~1 min)
# Needs: Setup, the Core cell (section 4) and section 8's experiment choice. No GPU, no UniMate.
#
# For each UniMate result, the same amplitude measures at each stage of the pipeline:
#   A. UniMate's own motion: its 19-joint rig, with fit_pupper's 40% shrink of
#      horizontal motion undone (that shrink is exactly invertible)
#   B. after the shrink (stride)
#   C. the final Pupper motion (after IK fit, stand blends, slow-down), measured on
#      the same rig (RigMap), so A, B and C are directly comparable
#   D. selection: of the 4 samples UniMate made, how active was the one we kept?
# Measures are ranges, not speeds, so time stretching doesn't change them.
# Keyframes (option 1) are shown as the reference for "how much a good motion moves".
import pickle

import pandas as pd

DIAG_OPTION = "3_unimate"  # @param ["3_unimate", "2_hybrid"]
# (2_hybrid's style results from before the hold fix are pinned to a stand: rerun them first.)

_res = pickle.loads(ACTIVE_FILE.read_bytes())  # the experiments picked in section 8
LEG_ORDER = list(SK["legs"])  # front_r, front_l, back_r, back_l
TRUNK_HEAD = [RIG_IX[n] for n in ("pelvis", "spine", "chest", "neck", "head")]


def rig_measures(P: np.ndarray) -> dict:
  """Amplitude of a rig motion P (T, 19, 3): ranges in degrees / cm."""
  roots = P[:, RIG_ROOTS]  # hips and shoulders, LEG_ORDER
  fr, fl, br, bl = (roots[:, LEG_ORDER.index(k)] for k in ("front_r", "front_l", "back_r", "back_l"))
  x = (fr + fl) / 2 - (br + bl) / 2  # body forward axis
  y = (fl + bl) / 2 - (fr + br) / 2  # body left axis
  pitch = np.degrees(np.arctan2(-x[:, 2], np.linalg.norm(x[:, :2], axis=1)))  # > 0: nose down
  roll = np.degrees(np.arctan2(y[:, 2], np.linalg.norm(y[:, :2], axis=1)))  # > 0: left side up
  base = roots.mean(1)
  feet = P[:, RIG_FEET]
  rel = P - base[:, None]
  exc = np.linalg.norm(rel - rel[:1], axis=-1).max(0)  # per joint: farthest from its start pose, relative to the body
  leg_joints = [j for j in range(P.shape[1]) if j not in TRUNK_HEAD]
  return dict(
    pitch_range_deg=float(np.ptp(pitch)),
    roll_range_deg=float(np.ptp(roll)),
    height_range_cm=100 * float(np.ptp(base[:, 2])),
    foot_lift_cm=100 * float((feet[..., 2] - feet[..., 2].min(0)).max()),
    foot_reach_cm=100 * float(max(np.linalg.norm(np.ptp(rel[:, j, :2], axis=0)) for j in RIG_FEET)),
    leg_excursion_cm=100 * float(exc[leg_joints].mean()),
    trunk_head_excursion_cm=100 * float(exc[TRUNK_HEAD].mean()),  # spine/neck/head: Pupper has none
  )


rows, sel = [], []
for r in _res:
  if r.get("motion") is None:
    continue
  final_rig = RIG.motion(motion_from_dict(r["motion"]))
  if r["option"] == "1_keyframes":
    rows.append(dict(request=r["request_id"], rep=r.get("rep", 0), stage="keyframes (reference)", **rig_measures(final_rig)))
    continue
  if r["option"] != DIAG_OPTION or r.get("skeleton") is None:
    continue
  stride = (r.get("config") or {}).get("stride", 0.6) if r["option"] == "3_unimate" else 1.0
  B = np.asarray(r["skeleton"], float)
  A = B.copy()
  A[..., :2] = B[:1, :, :2] + (B[..., :2] - B[:1, :, :2]) / stride
  for stage, P in (("A UniMate output", A), (f"B after stride {stride}", B), ("C fitted Pupper", final_rig)):
    rows.append(dict(request=r["request_id"], rep=r.get("rep", 0), stage=stage, **rig_measures(P)))
  rnd, idx = r.get("chosen", (0, 0))
  scores = r["sample_scores"][rnd]
  act = [s["activity_m_s"] for s in scores]
  sel.append(dict(request=r["request_id"], rep=r.get("rep", 0), passed=r["ok"], rounds=len(r["sample_scores"]),
                  chosen_activity=act[idx], max_activity=max(act),
                  activity_rank=1 + sorted(act, reverse=True).index(act[idx]),  # 1 = the most active sample
                  of=len(act)))

d = pd.DataFrame(rows)
cols = ["pitch_range_deg", "roll_range_deg", "height_range_cm", "foot_lift_cm", "foot_reach_cm",
        "leg_excursion_cm", "trunk_head_excursion_cm"]
print(f"Median amplitude per stage ({DIAG_OPTION}, all requests and repeats):")
order = ["keyframes (reference)"] + sorted(s for s in d.stage.unique() if s[0] in "ABC")
display(d.groupby("stage")[cols].median().reindex(order).round(1))

# How much of UniMate's motion survives each step: total over all results (sum of C / sum of A),
# which near-still results can't blow up. Trunk/head is left out: Pupper's torso is rigid, so it's always ~0.
body = [c for c in cols if c != "trunk_head_excursion_cm"]
tot = d[d.stage != "keyframes (reference)"].groupby("stage")[body].sum()
sA, sB, sC = sorted(s_ for s_ in d.stage.unique() if s_[0] in "ABC")
surv = pd.DataFrame({
  "stride (B/A)": tot.loc[sB] / tot.loc[sA].replace(0, np.nan),
  "IK fit + finishing (C/B)": tot.loc[sC] / tot.loc[sB].replace(0, np.nan),
  "end to end (C/A)": tot.loc[sC] / tot.loc[sA].replace(0, np.nan),
})
print("Fraction of UniMate's motion that survives (1.0 = all of it; above 1 = the step adds motion):")
display(surv.round(2))

s_ = pd.DataFrame(sel)
print("Selection: was the kept sample the most active of its 4 (rank 1) or the calmest (rank 4)?")
display(s_.groupby("activity_rank").size().rename("results").to_frame())
keep_ratio = float(np.median(s_.chosen_activity / s_.max_activity))
print(f"kept sample's activity / most active sample's: median {keep_ratio:.2f}")

ref = d[d.stage == "keyframes (reference)"][body].median()
gen = d[d.stage == sA][body].median()
rel_gen = float(np.nanmedian([gen[c] / ref[c] for c in body if ref[c] > 1e-6]))
rel_fit = float(np.nanmedian(surv["IK fit + finishing (C/B)"]))
rel_stride = float(np.nanmedian(surv["stride (B/A)"]))
rank = float(s_.activity_rank.mean())
trunk = float(d[d.stage == sA].trunk_head_excursion_cm.median())
print("\nReading it (each factor: 1.0 = no loss):")
print(f"- generation: UniMate's own motion is {rel_gen:.2f}x the keyframes' amplitude (median over measures)")
print(f"- stride shrink keeps {rel_stride:.2f} of it; IK fit + finishing keep {rel_fit:.2f}")
print(f"- selection keeps a sample with {keep_ratio:.2f}x the activity of the most active one (mean rank {rank:.1f} of 4)")
print(f"- plus {trunk:.1f} cm (median) of spine/neck/head motion in UniMate's output that Pupper has no joints for")
worst = min((rel_gen, "generation (UniMate itself moves little: try more steps, higher CFG, other prompts)"),
            (rel_stride, "the stride shrink (try stride 1.0 for in-place motions)"),
            (rel_fit, "the IK fit (retargeting: an orientation-aware, GMR-style fit)"),
            (keep_ratio, "selection (the ranking and gate prefer calm samples)"))
print(f"=> the biggest single loss: {worst[1]}")

## 11. Keep the results
Everything is already on Drive (`MyDrive/pupper_generation_benchmark`), unless Setup warned that Drive didn't mount. This zips all of it for a download.

In [ ]:
# @title Download everything (results, ratings, tuning tables)
import shutil

zip_path = shutil.make_archive("/content/pupper_generation_benchmark", "zip", RESULTS_DIR)
print("wrote", zip_path)
try:
  from google.colab import files

  files.download(zip_path)
except Exception as e:  # noqa: BLE001
  print("download it from the Files pane:", e)